# 第6章 ダイオードクリッパとニュートン法モデル（ノートブック版）

教科書の第6章のうち、**Pico がなくても動く部分**（6-6-2 の `clipper_model.py`・`newton_demo.py`・`synth_clipper.py`・`clipper_analysis.py`）を、本文の `run_ch6.sh synth`（と `analyze`）の流れのとおりに1冊にまとめたものです。上から順に実行してください（Mac の Jupyter でも Google Colab でも動きます。執筆時の Mac で全部で約6分。Colab ではその数倍かかることがあります）。

| 節 | 試すこと | 教科書 | 使うファイル |
|---|---|---|---|
| 1 | ニュートン法の収束（初期値と方法、2 次収束、照合、Rs の効き方、台形則の初期値） | 6-3-4、手順 A の 1 | `newton_demo.py` |
| 2 | 6-3-1 の数値例（頭打ちの電圧と THD）と、IV-8 の台形則との照合 | 6-3-1・6-3-4 の (5) | `clipper_model.py` |
| 3 | 合成データを作る（Pico B の取り込みの代わり。44 回分） | 手順 A の 1 | `synth_clipper.py` |
| 4 | 合成データの解析：雑音・τ の校正・THD をモデルの区間と比べる | 手順 A の 1・E の 10〜11、6-7 | `clipper_analysis.py` |
| 4-2 | 取り込みの始まりの位相がばらばらな合成データ（実機と同じ）で、位相をそろえる処理を確かめる | 6-3-3、手順 A の 1 | `synth_clipper.py --random-phase`・`clipper_analysis.py` |
| 5 | **第1章の事後サンプルが無いとき**（IV-8 の仮定の分布 `prior_samples`） | 6-7 の (c)、`run_ch6.sh analyze` | `clipper_analysis.py` |
| 6 | ADC_VREF の目盛りの不確かさを伝播させる（`--vref-rel`） | 6-3-6 | `clipper_analysis.py` |
| 7 | （任意。既定ではとばす）条件を変えたとき：D2 を D1 で代用・温度の補正なし・ADC_VREF のずれ | 6-7 の (a)(b)、6-3-6 | `clipper_analysis.py` |

- 次のセルたちが、教科書と同じフォルダの形（`home-lab/ch6/` と `home-lab/partIV/common/`）を、このノートブックの隣に作り、6-6 のファイルと IV-8 の `common/nonlinear.py` を `%%writefile` で書き出します（第6章のファイルは第IV部のファイルが無くても動きます。ここでは、`newton_demo.py` の照合と 2 節で IV-8 の関数とも比べるために、IV-8 のものも書き出します。IV-8 のものが元で、第6章には写しを置きません）。中身は教科書と同じです。変えたのは、図の日本語フォントの候補を足したこと（`clipper_analysis.py` の `matplotlib.rcParams["font.family"]` の行）だけです。
- 各節では、`run_ch6.sh` の `uv run python ○○.py …` の代わりに、`run("○○.py …")` で同じ引数のまま実行します（`home-lab/ch6/` の中で `python ○○.py …` を別のプロセスで動かします）。表示を見てから、`ok()` で教科書の手順 A の 1 と 6-7 の「確かめること」を確かめます。乱数の種は教科書と同じなので、値は 6-7 の表と同じになります。
- **入れていないもの（Pico B が要る部分）**：`clipper.c` のビルドと書き込み（手順 B の 4）、配線と `run_ch6.sh check`・`run`（`clipper_capture.py`。手順 C〜D）、第1章で 1N4148 の A・B を測って事後サンプル（`*_samples.npz`）を作ること（手順 A の 2）。ここでは、Pico B の取り込みのかわりに `synth_clipper.py` の合成データを、第1章の事後サンプルのかわりに `synth_clipper.py` が作る合成の事後サンプル（4 節）と、IV-8 の仮定の分布（5 節）を使います。`clipper.c` は、最後に読むためだけに載せてあります（ビルドはしません）。
- 自分の取り込み（`data/run1/`）を解析するときは、Mac で教科書の手順 E のとおりに `bash run_ch6.sh analyze run1` を行ってください。
- ⚠ 実物の注意（OFF の後 15 秒待ってから USB を抜く、C_B 470µF の向きと放電、ADC のピンにかかる電圧）は、教科書の 6-4 を読んでください。このノートブックは計算だけです。
- Colab では、最初のほうの「0-1. 図の日本語のフォント」のセルが日本語のフォント（Noto Sans CJK JP）を入れます（数十秒〜1分ほど。Mac では Hiragino Sans を使うので何もしません）。そのセルが「★日本語のフォントが見つからない」と表示したときは、図の日本語が四角（□）になります（数値の確認には影響しません）。

In [ ]:
# 0. 準備：使うパッケージ（numpy・scipy・pandas・matplotlib。Colab には最初から入っている。版は要確認）
import numpy, scipy, pandas, matplotlib, sys
print("OK", "Python", sys.version.split()[0], "numpy", numpy.__version__, "scipy", scipy.__version__,
      "pandas", pandas.__version__, "matplotlib", matplotlib.__version__)
print("（教科書は Python 3.12.15・numpy 2.5.5・scipy 1.18.1・pandas 3.1.1・matplotlib 3.11.2 で確かめた。pyserial は Pico B と話すときだけ使うので、ここでは要らない）")

In [ ]:
# 0-1. 準備：図の日本語のフォント（Colab のときだけ fonts-noto-cjk を入れる。Mac は Hiragino Sans を使うので何もしない）
import glob, os, subprocess, sys
import matplotlib
from matplotlib import font_manager

JP_FONTS = ["Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic"]   # 本文の図と同じ候補（上から順に使う）


def find_jp_font():
    have = {f.name for f in font_manager.fontManager.ttflist}
    return next((n for n in JP_FONTS if n in have), None)


IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and find_jp_font() is None:
    print("Colab：日本語のフォント（fonts-noto-cjk）を入れます（数十秒〜1分ほど）")
    cmd = ["apt-get", "install", "-y", "-qq", "fonts-noto-cjk"]
    if subprocess.run(cmd, capture_output=True).returncode != 0:
        subprocess.run(["apt-get", "update", "-qq"], capture_output=True)   # パッケージの一覧が古いときは、更新してもう一度
        subprocess.run(cmd, capture_output=True)
    for path in glob.glob("/usr/share/fonts/opentype/noto/NotoSansCJK*.ttc"):
        font_manager.fontManager.addfont(path)   # このノートブックの中でかく図に効く（再起動は要らない）
    # 別のプロセスで動かすスクリプト（subprocess や !python）にも見えるように、フォントの一覧のキャッシュを書き直す
    font_manager.json_dump(font_manager.fontManager, os.path.join(
        matplotlib.get_cachedir(), f"fontlist-v{font_manager.FontManager.__version__}.json"))

JP_FONT = find_jp_font()
if JP_FONT:
    matplotlib.rcParams["font.family"] = [JP_FONT, "sans-serif"]
    matplotlib.rcParams["font.sans-serif"] = [JP_FONT] + [f for f in matplotlib.rcParams["font.sans-serif"] if f != JP_FONT]
    if IN_COLAB:   # 本文のスクリプトの "sans-serif" も日本語のフォントになるように、Colab の matplotlib の設定ファイルに書く
        rc = os.path.join(matplotlib.get_configdir(), "matplotlibrc")
        lines = open(rc, encoding="utf-8").read().splitlines() if os.path.exists(rc) else []
        lines = [l for l in lines if not l.startswith("font.sans-serif")]
        lines.append("font.sans-serif: " + ", ".join(matplotlib.rcParams["font.sans-serif"]))
        with open(rc, "w", encoding="utf-8") as f:
            f.write("\n".join(lines) + "\n")
    print("図の日本語のフォント：", JP_FONT)
else:
    print("★日本語のフォントが見つからないので、図の日本語が四角（□）になります（数値には影響しません）。")
    print("  Colab なら、インターネットにつながっているかを確かめて、このセルをもう一度実行してください。"
          "Linux の Jupyter なら、sudo apt install fonts-noto-cjk で入れてから、カーネルを再起動します。")

次のセルで、教科書と同じフォルダの形を作ります。続く6つのセルで、IV-8 の `partIV/common/`（`__init__.py`・`nonlinear.py`。照合の相手で、第6章だけなら無くてよい）と 6-6 の4つのファイルを書き出します。

In [ ]:
import os
os.makedirs("home-lab/ch6", exist_ok=True)
os.makedirs("home-lab/partIV/common", exist_ok=True)
print("home-lab/ch6/ と home-lab/partIV/common/ を作りました")

In [ ]:
%%writefile home-lab/partIV/common/__init__.py
"""第IV部の共通ライブラリ（stft・audio_io・plotting。後の章で metrics などを足す）"""

In [ ]:
%%writefile home-lab/partIV/common/nonlinear.py
"""第IV部の共通ライブラリ：非線形システムとダイオードクリッパ（IV-8 で定義）

  静的な非線形（記憶のない非線形）
    y = soft_clip(x, drive=1.0)          tanh(drive·x)/tanh(drive)（|x| ≤ 1 を |y| ≤ 1 に）
    y = hard_clip(x, c=0.5)              ±c で止める
  ダイオードクリッパ（直列抵抗 R → 出力にコンデンサ C と逆並列のダイオード 2 本。図IV-8-19）
    C dv/dt = (vin − v)/R − 2 Is sinh(v/(n VT))
    p = clipper_params(R=2.2e3, C=10e-9, Is=2.52e-9, n=1.752, VT=0.02585)
    v = clipper_static_newton(vin, p)    C = 0（記憶なし）のときの v（ニュートン法、ダイオード 2 本）
    v = clipper_static_lambertw(vin, p)  ダイオード 1 本で近似した陽な解（Lambert W。符号で向きを選ぶ）
    v = clipper_forward_euler(vin, fs, p)                陽解法（前進オイラー）
    v, it = clipper_backward_euler(vin, fs, p)           後退オイラー ＋ ニュートン法（it は反復回数）
    v, it = clipper_trapezoid(vin, fs, p)                台形則（双一次変換と同じ）＋ ニュートン法
    t, v = clipper_reference(f_vin, t_end, fs_out, p)    scipy の solve_ivp（Radau）で細かく解いた基準
    y = oversampled(x, func, factor)                     factor 倍に上げて func を通し、元に戻す
  ひずみの測り方
    h = harmonic_amps(x, fs, f0, n_harm)   最小二乗で k·f0 の振幅を求める（h[0] が基本波）
    r = thd(x, fs, f0, n_harm=20)          THD_F = √(Σ_{k≥2} A_k²)/A_1（基本波に対する比）
    r = thd_fft(x, fs, f0, n_harm=20)      同じものを FFT のビンで（整数周期を切り出す）
    db = inharmonic_db(x, fs, f0)          k·f0 以外の成分のパワーの割合 [dB]（折り返しの目安）
"""
import numpy as np

__all__ = ["soft_clip", "hard_clip", "clipper_params", "diode_i", "diode_g",
           "clipper_static_newton", "clipper_static_lambertw",
           "clipper_forward_euler", "clipper_backward_euler", "clipper_trapezoid",
           "clipper_reference", "oversampled", "harmonic_amps", "thd", "thd_fft", "inharmonic_db"]


# ---------------------------------------------------------------- 静的な非線形
def soft_clip(x, drive=1.0):
    """tanh 型のソフトクリップ。x = ±1 で ±1 になるように tanh(drive) で割る"""
    return np.tanh(drive * np.asarray(x, dtype=float)) / np.tanh(drive)


def hard_clip(x, c=0.5):
    """ハードクリップ：|x| > c を ±c で止める"""
    return np.clip(np.asarray(x, dtype=float), -c, c)


# ---------------------------------------------------------------- ダイオードクリッパ
def clipper_params(R=2.2e3, C=10e-9, Is=2.52e-9, n=1.752, VT=0.02585):
    """回路の定数（見本の値。Is・n は 1N4148 の SPICE モデルでよく使われる値で、データシートで要確認。
    第1章でダイオードを測っていれば、その推定値に置き換える）。nVT = n·VT [V]"""
    return dict(R=float(R), C=float(C), Is=float(Is), nVT=float(n * VT))


def diode_i(v, p):
    """逆並列のダイオード 2 本の電流 Is(e^{v/nVT} − 1) − Is(e^{−v/nVT} − 1) = 2 Is sinh(v/nVT)"""
    return 2.0 * p["Is"] * np.sinh(v / p["nVT"])


def diode_g(v, p):
    """その微分（小信号のコンダクタンス）2 Is cosh(v/nVT)/nVT"""
    return 2.0 * p["Is"] * np.cosh(v / p["nVT"]) / p["nVT"]


def _newton_scalar(fun, dfun, v0, tol=1e-12, max_iter=50, max_step=None):
    """1 変数のニュートン法。max_step を与えると、1 回に動く量をその大きさで止める（指数が大きすぎる点に
    飛んであふれるのを防ぐ。SPICE の「電圧の制限」と同じ考え方）。戻り値 (v, 反復回数)"""
    v = v0
    for it in range(1, max_iter + 1):
        step = fun(v) / dfun(v)
        if max_step is not None:
            step = np.clip(step, -max_step, max_step)
        v -= step
        if abs(step) < tol:
            return v, it
    raise RuntimeError(f"ニュートン法が {max_iter} 回で収束しない（v = {v}）")


def clipper_static_newton(vin, p, tol=1e-13):
    """C = 0 のとき (vin − v)/R = 2 Is sinh(v/nVT) を v について解く（各点ごとのニュートン法）"""
    vin = np.atleast_1d(np.asarray(vin, dtype=float))
    out = np.empty_like(vin)
    R = p["R"]
    for i, u in enumerate(vin):
        f = lambda v: v - u + R * diode_i(v, p)
        df = lambda v: 1.0 + R * diode_g(v, p)
        v0 = np.clip(u, -0.8, 0.8)                   # ダイオードの順電圧より大きくはならない
        out[i], _ = _newton_scalar(f, df, v0, tol=tol, max_step=4 * p["nVT"])
    return out


def clipper_static_lambertw(vin, p):
    """ダイオード 1 本（vin > 0 なら順方向の 1 本だけ）で近似した陽な解。
    (vin − v)/R = Is(e^{v/nVT} − 1) を解くと
        v = vin + R Is − nVT·W( (R Is/nVT)·exp((vin + R Is)/nVT) )
    W の中身はすぐ巨大になるので、W(e^z) = ω(z)（Wright の ω 関数、scipy.special.wrightomega）を使う：
        z = ln(R Is/nVT) + (vin + R Is)/nVT。vin < 0 は奇関数として −v(−vin)"""
    from scipy.special import wrightomega
    vin = np.asarray(vin, dtype=float)
    s = np.sign(vin)
    u = np.abs(vin)
    R, Is, V = p["R"], p["Is"], p["nVT"]
    z = np.log(R * Is / V) + (u + R * Is) / V
    v = u + R * Is - V * np.real(wrightomega(z))
    return s * v


def clipper_forward_euler(vin, fs, p, v0=0.0):
    """前進オイラー（陽解法）：v[n] = v[n−1] + h·f(v[n−1], vin[n−1])。h が大きいと発散する"""
    h = 1.0 / fs
    R, C = p["R"], p["C"]
    v = np.empty(len(vin))
    vp = v0
    with np.errstate(over="ignore", invalid="ignore"):
        for n in range(len(vin)):
            u_prev = vin[n - 1] if n > 0 else 0.0
            vp = vp + h * ((u_prev - vp) / R - diode_i(vp, p)) / C
            v[n] = vp
    return v


def clipper_backward_euler(vin, fs, p, v0=0.0, tol=1e-10, max_iter=50):
    """後退オイラー（陰解法）：v[n] = v[n−1] + h·f(v[n], vin[n])。毎時刻、
        g(v) = v − v[n−1] − (h/C)·((vin[n] − v)/R − 2 Is sinh(v/nVT)) = 0
    をニュートン法で解く（g'(v) = 1 + (h/C)(1/R + 2 Is cosh(v/nVT)/nVT) > 0 なので解は1つ）。
    戻り値 (v, 各時刻のニュートン法の反復回数)"""
    h = 1.0 / fs
    R, C = p["R"], p["C"]
    k = h / C
    v = np.empty(len(vin))
    its = np.empty(len(vin), dtype=int)
    vp = v0
    for n, u in enumerate(vin):
        g = lambda x: x - vp - k * ((u - x) / R - diode_i(x, p))
        dg = lambda x: 1.0 + k * (1.0 / R + diode_g(x, p))
        vp, its[n] = _newton_scalar(g, dg, vp, tol=tol, max_iter=max_iter, max_step=4 * p["nVT"])
        v[n] = vp
    return v, its


def clipper_trapezoid(vin, fs, p, v0=0.0, tol=1e-10, max_iter=50):
    """台形則：v[n] = v[n−1] + (h/2)(f(v[n−1], vin[n−1]) + f(v[n], vin[n]))（2 次の精度。
    線形の部分については IV-1 の双一次変換と同じ写像になる）。ニュートン法で解く"""
    h = 1.0 / fs
    R, C = p["R"], p["C"]
    k = h / (2.0 * C)
    v = np.empty(len(vin))
    its = np.empty(len(vin), dtype=int)
    vp, up = v0, 0.0
    for n, u in enumerate(vin):
        fp = (up - vp) / R - diode_i(vp, p)                  # 1 つ前の時刻の f（分かっている）
        g = lambda x: x - vp - k * (fp + (u - x) / R - diode_i(x, p))
        dg = lambda x: 1.0 + k * (1.0 / R + diode_g(x, p))
        vp, its[n] = _newton_scalar(g, dg, vp, tol=tol, max_iter=max_iter, max_step=4 * p["nVT"])
        v[n] = vp
        up = u
    return v, its


def clipper_reference(f_vin, t_end, fs_out, p, rtol=1e-10, atol=1e-13):
    """基準の解：scipy.integrate.solve_ivp（Radau：硬い方程式向けの陰的ルンゲ–クッタ法）で、
    入力を連続時間の関数 f_vin(t) として与え、時刻 n/fs_out の値を返す"""
    from scipy.integrate import solve_ivp
    R, C = p["R"], p["C"]
    rhs = lambda t, v: ((f_vin(t) - v) / R - diode_i(v, p)) / C
    jac = lambda t, v: np.array([[-(1.0 / R + diode_g(v[0], p)) / C]])
    t = np.arange(int(round(t_end * fs_out))) / fs_out
    sol = solve_ivp(rhs, (0.0, t[-1]), [0.0], method="Radau", t_eval=t, rtol=rtol, atol=atol, jac=jac,
                    max_step=0.25 / fs_out)
    if not sol.success:
        raise RuntimeError(sol.message)
    return t, sol.y[0]


def oversampled(x, func, factor=8):
    """x を factor 倍の標本化周波数に上げ（resample_poly）、func を通し、元の標本化周波数に戻す。
    上げるときの低域通過は「像」を消し、戻すときの低域通過は、非線形が作った高い倍音のうち
    元の fs/2 を超える分を、折り返す前に消す（IV-1-3-3）。func は (x_up, fs倍率) を受け取らず x_up だけを受け取る"""
    from scipy.signal import resample_poly
    if factor == 1:
        return func(np.asarray(x, dtype=float))
    xu = resample_poly(x, factor, 1)
    yu = func(xu)
    return resample_poly(yu, 1, factor)


# ---------------------------------------------------------------- ひずみの測り方
def harmonic_amps(x, fs, f0, n_harm=20):
    """x ≈ c + Σ_k (a_k cos 2πk f0 t + b_k sin 2πk f0 t) を最小二乗で当てはめ、
    振幅 A_k = √(a_k² + b_k²)（k = 1, …, n_harm。fs/2 以上の k は 0）を返す"""
    x = np.asarray(x, dtype=float)
    t = np.arange(len(x)) / fs
    ks = [k for k in range(1, n_harm + 1) if k * f0 < fs / 2]
    cols = [np.ones_like(t)]
    for k in ks:
        cols += [np.cos(2 * np.pi * k * f0 * t), np.sin(2 * np.pi * k * f0 * t)]
    coef, *_ = np.linalg.lstsq(np.stack(cols, axis=1), x, rcond=None)
    A = np.zeros(n_harm)
    for i, k in enumerate(ks):
        A[k - 1] = np.hypot(coef[1 + 2 * i], coef[2 + 2 * i])
    return A


def thd(x, fs, f0, n_harm=20):
    """THD_F = √(A_2² + … + A_K²)/A_1（基本波に対する比。IEEE などの「F」の定義）。
    THD_R（全体の実効値に対する比）は THD_F/√(1 + THD_F²)"""
    A = harmonic_amps(x, fs, f0, n_harm)
    return float(np.sqrt(np.sum(A[1:] ** 2)) / A[0])


def thd_fft(x, fs, f0, n_harm=20):
    """FFT で THD_F を求める。f0 の周期がちょうど整数個入る長さ（fs/f0 が整数のとき）に切り、
    方形窓で FFT すると、k 倍音はビン k·(周期の数) に漏れなく入る（コヒーレント・サンプリング）"""
    P = fs / f0
    if abs(P - round(P)) > 1e-9:
        raise ValueError("fs/f0 が整数でない（整数周期を切り出せない）。thd() を使う")
    P = int(round(P))
    ncyc = len(x) // P
    seg = np.asarray(x[:ncyc * P], dtype=float)
    S = np.abs(np.fft.rfft(seg)) * 2 / len(seg)
    A = np.array([S[k * ncyc] if k * ncyc < len(S) - 1 else 0.0 for k in range(1, n_harm + 1)])
    return float(np.sqrt(np.sum(A[1:] ** 2)) / A[0])


def inharmonic_db(x, fs, f0, n_harm=None, width_hz=None):
    """k·f0（k = 0, 1, 2, …）の近く（±width_hz）以外のパワーの、全体に対する割合 [dB]。
    非線形が作る倍音のうち fs/2 を超えたものは、k·f0 でない周波数に折り返すので、これで折り返しを測る。
    窓は blackman-harris（漏れが小さい）、width_hz の既定は 6 ビン分"""
    from scipy.signal import get_window
    x = np.asarray(x, dtype=float)
    w = get_window("blackmanharris", len(x))
    S = np.abs(np.fft.rfft(x * w)) ** 2
    f = np.fft.rfftfreq(len(x), 1 / fs)
    width_hz = width_hz or 6 * fs / len(x)
    k = np.round(f / f0)
    near = np.abs(f - k * f0) <= width_hz
    return float(10 * np.log10(S[~near].sum() / S.sum()))

In [ ]:
%%writefile home-lab/ch6/clipper_model.py
"""第6章：ダイオードクリッパのモデルと解析の道具（~/home-lab/ch6/clipper_model.py）

  回路（図6-1）：vin ─ R ─ v ─（C ∥ D1 ∥ D2）─ Vb（バイアス。交流では GND と同じ）
      C dv/dt = (vin − v)/R − iD(v)、  iD(v) = i(v; D1) − i(−v; D2)        （★。v・vin は Vb から測る）
  ダイオード 1 本は第1章のショックレー式＋直列抵抗：i = Is[exp((V − i Rs)/nVT) − 1]
  （Lambert W で陽に書く。第1章 1-3-2）。IV-8 の common/nonlinear.py は Rs = 0・2 本同じとしたもの。
  このファイルは numpy・scipy だけを使い、第IV部のファイルは import しない（第6章を第IV部より先に進められる）。

  d = diode(log10_Is, n, Rs, temp_c=25.0, shift=0.0)   ダイオードの定数（配列でもよい：事後分布のサンプルを並べる）
  i, g = diode_iv(V, d)                                 電流と、その微分（小信号のコンダクタンス）
  i, g = pair_iv(v, d1, d2)                             逆並列の 2 本
  v, it, hist = newton_static(u, R, d1, d2, v0, mode)   C = 0 のときの (u − v)/R = iD(v)。mode は plain/limit/line
  v, st = simulate_periodic(vin, f0, R, C, d1, d2, up=4)  1 周期分の vin（等間隔 N 点、周期的）を入れたときの定常解
  A = harmonics(x, kmax=20)                             1 周期 N 点の波形の k 倍音の振幅（k = 0…kmax）
  thd_f(x, kmax=20)                                     THD_F（基本波に対する比。IV-8 と同じ定義）
  S = load_posterior(path, size, rng)                   第1章の事後サンプル（*_samples.npz）から size 組
  S = prior_samples(size, rng)                          事後サンプルが無いときの代わり（IV-8 の発展課題の仮定）
"""
import numpy as np
from scipy.special import wrightomega


KB, Q = 1.380649e-23, 1.602176634e-19
__all__ = ["vt", "diode", "diode_iv", "pair_iv", "newton_static", "simulate_periodic", "harmonics",
           "thd_f", "load_posterior", "prior_samples"]


def vt(temp_c):
    """熱電圧 kT/q [V]"""
    return KB * (np.asarray(temp_c, dtype=float) + 273.15) / Q


def diode(log10_Is, n, Rs, temp_c=25.0, shift=0.0):
    """ダイオード 1 本の定数。値は数でも配列（事後サンプル）でもよい。
    shift [V]：温度の違いの補正。測ったときより温度が ΔT 高いと、同じ電流の電圧は約 κΔT（κ ≈ −2 mV/℃、
    データシートで要確認）下がるので、i(V) を i(V + shift)、shift = −κΔT として使う"""
    f = lambda x: np.asarray(x, dtype=float)
    return dict(Is=10.0 ** f(log10_Is), a=f(n) * vt(temp_c), Rs=f(Rs), shift=f(shift))


def diode_iv(V, d):
    """i = Is[exp((V − iRs)/a) − 1]（a = nVT）の電流 i と di/dV。Rs > 0 は Lambert W（Wright の ω 関数）で：
        x = ln(Is Rs/a) + (V + Is Rs)/a、w = ω(x) = W(e^x)、i = (a/Rs) w − Is、di/dV = w / (Rs (1 + w))
    Rs = 0 はショックレー式そのまま（V が大きいと exp があふれて inf になる。newton_demo.py の 1.）"""
    V = np.asarray(V, dtype=float) + d["shift"]
    Is, a, Rs = d["Is"], d["a"], d["Rs"]
    if np.all(Rs <= 0):
        with np.errstate(over="ignore"):
            e = np.exp(V / a)
        return Is * (e - 1.0), Is * e / a
    Rs = np.maximum(Rs, 1e-9)
    x = np.log(Is * Rs / a) + (V + Is * Rs) / a
    w = np.real(wrightomega(x))
    return a / Rs * w - Is, w / (Rs * (1.0 + w))


def pair_iv(v, d1, d2):
    """逆並列の 2 本：D1 は v > 0 で順方向、D2 は v < 0 で順方向。iD = i1(v) − i2(−v)、g = g1(v) + g2(−v)"""
    i1, g1 = diode_iv(v, d1)
    i2, g2 = diode_iv(-np.asarray(v, dtype=float), d2)
    return i1 - i2, g1 + g2


def newton_static(u, R, d1, d2, v0=0.0, mode="line", tol=1e-12, max_iter=200):
    """C = 0 のとき f(v) = v − u + R iD(v) = 0 を解く（f は単調増加なので解は 1 つ）。
      plain：v ← v − f/f'（そのまま）
      limit：1 回に動く量を ±4nVT までに制限（IV-8 の nonlinear.py と同じ。SPICE の電圧の制限）
      line ：減衰つきニュートン法。|f| が小さくならなければ、動く量を半分にしてやり直す（直線探索）
    戻り値 (v, 反復回数, 各回の v のリスト)。収束しなければ反復回数は −1"""
    v = float(v0)
    hist = [v]
    amax = 4.0 * float(np.max(d1["a"]))
    with np.errstate(over="ignore", invalid="ignore"):
        for it in range(1, max_iter + 1):
            i, g = pair_iv(v, d1, d2)
            f = v - u + R * i
            step = -f / (1.0 + R * g)
            if not np.isfinite(step):
                return v, -1, hist
            if mode == "limit":
                step = float(np.clip(step, -amax, amax))
            elif mode == "line":
                lam = 1.0
                while lam > 1e-6:
                    i2, _ = pair_iv(v + lam * step, d1, d2)
                    f2 = v + lam * step - u + R * i2
                    if np.isfinite(f2) and abs(f2) < abs(f):
                        break
                    lam *= 0.5
                step *= lam
            v += float(step)
            hist.append(v)
            if abs(step) < tol:
                return v, it, hist
    return v, -1, hist


def _fft_upsample(x, up):
    """周期的で帯域が限られた 1 周期 N 点を、up 倍の点数に補間する（FFT のゼロ詰め。周期の端もなめらか）"""
    N = x.shape[-1]
    X = np.fft.rfft(x, axis=-1)
    X[..., -1] *= 0.5                                   # ナイキストのビンは 2 つに分かれるので半分
    Y = np.zeros(x.shape[:-1] + (N * up // 2 + 1,), dtype=complex)
    Y[..., :X.shape[-1]] = X
    return np.fft.irfft(Y, n=N * up, axis=-1) * up


def simulate_periodic(vin, f0, R, C, d1, d2, up=4, max_periods=12, tol_period=1e-7, tol=1e-11):
    """（★）を台形則＋ニュートン法で解き、周期的な定常解を返す（IV-8 の clipper_trapezoid と同じ式）。
    vin：1 周期 N 点（等間隔。Vb から測った電圧。(S, N) なら組ごとに別の入力）、f0：基本周波数 [Hz]。R・C・d1・d2 の値は数でも長さ S の配列でもよく、
    配列なら S 組のパラメータを同時に計算する（事後サンプルを一度に通すため。ベクトル化）。
    up 倍に補間した時刻で解き（刻み h = 周期/(N·up)）、元の N 点の時刻の値を返す。
    1 周期ずつ進め、周期の初めと終わりの差が tol_period [V] 未満になったら定常とみなす。
    戻り値 (v（S, N）または（N,）, 情報の辞書：周期の数・ニュートン法の反復の平均と最大)"""
    vin = np.asarray(vin, dtype=float)
    N = vin.shape[-1]
    S = int(np.broadcast(np.asarray(R), np.asarray(C), d1["Is"], d1["a"], d1["Rs"], d2["Is"], d2["a"],
                         vin[..., 0]).size)
    R = np.broadcast_to(np.asarray(R, dtype=float), (S,))
    C = np.broadcast_to(np.asarray(C, dtype=float), (S,))
    u = _fft_upsample(vin, up)
    M = N * up
    k = 1.0 / (f0 * M) / 2.0                           # h/2（h は刻みの時間 [s]）
    v = np.zeros(S)
    out = np.empty((S, M))
    its = []
    amax = 4.0 * np.max(d1["a"])
    prev_start = None
    for per in range(1, max_periods + 1):
        start = v.copy()
        up_ = u[..., -1]
        for m in range(M):
            iD, _ = pair_iv(v, d1, d2)
            fp = ((up_ - v) / R - iD) / C               # 1 つ前の時刻の右辺（分かっている）
            x = v.copy()
            for it in range(1, 60):
                iDx, gx = pair_iv(x, d1, d2)
                g = x - v - k * (fp + ((u[..., m] - x) / R - iDx) / C)
                dg = 1.0 + k * (1.0 / R + gx) / C
                step = np.clip(g / dg, -amax, amax)
                x = x - step
                if np.max(np.abs(step)) < tol:
                    break
            its.append(it)
            v = x
            out[:, m] = v
            up_ = u[..., m]
        if prev_start is not None and np.max(np.abs(v - start)) < tol_period:
            break
        prev_start = start
    res = out[:, ::up]
    info = dict(periods=per, newton_mean=float(np.mean(its)), newton_max=int(np.max(its)),
                periodic_err=float(np.max(np.abs(v - start))))
    return (res[0] if S == 1 else res), info


def harmonics(x, kmax=20):
    """1 周期ちょうど N 点の波形の k 倍音の振幅 A_k（k = 0…kmax。A_0 は直流）。
    整数周期を方形窓で FFT すると、k 倍音はビン k にちょうど乗る（漏れがない）"""
    x = np.asarray(x, dtype=float)
    N = x.shape[-1]
    X = np.fft.rfft(x, axis=-1) / N
    A = 2.0 * np.abs(X[..., :kmax + 1])
    A[..., 0] /= 2.0
    return A


def thd_f(x, kmax=20):
    """THD_F = √(A_2² + … + A_kmax²) / A_1"""
    A = harmonics(x, kmax)
    return np.sqrt(np.sum(A[..., 2:] ** 2, axis=-1)) / A[..., 1]


def load_posterior(path, size, rng):
    """第1章の iv_analysis.py が保存した *_samples.npz（log10_Is・n・Rs・temp_C）から size 組を取り出す。
    3 つを同じ行から取る（組のまま）ことが大事：log10 Is と n は強く相関している（第1章 1-3-4）"""
    z = np.load(path)
    m = len(z["n"])
    idx = rng.choice(m, size=size, replace=size > m)
    return dict(log10_Is=z["log10_Is"][idx], n=z["n"][idx], Rs=z["Rs"][idx], temp_C=float(z["temp_C"]))


def prior_samples(size, rng):
    """事後サンプルが無いときの代わり（IV-8 の発展課題と同じ仮定。1N4148 の SPICE モデルの見本の値のまわり）：
    log10 Is ~ N(log10 2.52e-9, 0.3²)、n ~ N(1.75, 0.05²)（独立）、Rs ~ 対数正規（中央 0.5Ω、ln の sd 1）。
    これは「測っていない」ことを表す広い分布で、予測の区間も広くなる"""
    return dict(log10_Is=rng.normal(np.log10(2.52e-9), 0.3, size), n=rng.normal(1.75, 0.05, size),
                Rs=np.exp(rng.normal(np.log(0.5), 1.0, size)), temp_C=25.0)

In [ ]:
%%writefile home-lab/ch6/newton_demo.py
"""第6章：ニュートン法の収束を確かめる（~/home-lab/ch6/newton_demo.py）。実機は要らない

  1. C = 0 の式 f(v) = v − u + R iD(v) = 0 を、初期値と方法（plain・limit・line）を変えて解き、反復回数を比べる
  2. 2 次収束：誤差 e_k が e_{k+1} ≈ K e_k² で小さくなることを数値で見る（K = f''/(2f')）
  3. 照合：scipy.optimize.brentq（区間を半分にしていく方法）と、IV-8 の nl.clipper_static_newton（Rs = 0）。
     IV-8 の ~/home-lab/partIV/common/nonlinear.py が無ければ、IV-8 との照合だけをとばす（brentq との照合はいつも行う）
  4. Rs の効き方：Rs = 0・1・10 Ω で出力がどれだけ変わるか（この回路の電流は 0.5 mA 以下）
  5. C があるとき（台形則）：毎時刻の初期値を「1 つ前の値」にするか「0」にするかで、反復回数がどう変わるか
"""
import pathlib
import sys

import numpy as np
from scipy.optimize import brentq

import clipper_model as cm

sys.path.insert(0, str(pathlib.Path(__file__).resolve().parents[1] / "partIV"))
try:
    from common import nonlinear as nl  # IV-8 の共通ライブラリ（あれば照合の相手にする）
except ImportError:
    nl = None                           # 第IV部をまだ進めていない：無くてもこのファイルは最後まで動く
R = 2.2e3
d = cm.diode(np.log10(2.52e-9), 1.752, 0.0, temp_c=25.0)       # IV-8 の見本の値（Rs = 0）
a = float(d["a"])
f = lambda v, u: v - u + R * float(cm.pair_iv(v, d, d)[0])
print(f"見本の値：R = {R:.0f} Ω、Is = 2.52 nA、n = 1.752、25℃（nVT = {a * 1e3:.2f} mV）")

print("\n1. 初期値と方法による反復回数（−1 は失敗：値があふれた・200 回で止まらない）")
print("    u [V]   v0 [V]    plain   limit   line      解 v [V]")
for u in (0.3, 1.0, 3.0):
    for v0 in (0.0, u, 2.0, 40.0):
        r = [cm.newton_static(u, R, d, d, v0=v0, mode=m) for m in ("plain", "limit", "line")]
        sol = next((x[0] for x in r if x[1] > 0), float("nan"))
        print(f"    {u:4.1f}   {v0:6.1f}   {r[0][1]:6d}  {r[1][1]:6d}  {r[2][1]:6d}    {sol:.6f}")

print("\n2. 2 次収束（u = 1 V、v0 = 0、line）：e_k = |v_k − v*|")
v_star, _, hist = cm.newton_static(1.0, R, d, d, v0=0.0, mode="line", tol=1e-15)
e = np.abs(np.array(hist) - v_star)
i_s, g_s = cm.pair_iv(v_star, d, d)
fp = 1 + R * float(g_s)
fpp = R * 2 * float(d["Is"]) * np.sinh(v_star / a) / a ** 2           # Rs = 0 の 2 階微分
print(f"    K = f''/(2f') = {fpp / (2 * fp):.2f} /V")
for k in range(len(e) - 1):
    if e[k] == 0:
        break
    ratio = e[k + 1] / e[k] ** 2 if e[k + 1] > 0 else float("nan")
    print(f"    k = {k:2d}   e_k = {e[k]:.3e}   e_(k+1)/e_k² = {ratio:9.3f}")

print("\n3. 照合（u = −3〜3 V の 121 点）")
us = np.linspace(-3, 3, 121)
mine = np.array([cm.newton_static(u, R, d, d, mode="line")[0] for u in us])
br = np.array([brentq(lambda v: f(v, u), -1.0, 1.0, xtol=1e-15) for u in us])
if nl is not None:
    iv8 = nl.clipper_static_newton(us, nl.clipper_params(R=R, Is=2.52e-9, n=1.752, VT=float(cm.vt(25.0))))
    print(f"    brentq との差の最大 {np.max(np.abs(mine - br)):.1e} V、IV-8 の clipper_static_newton との差の最大 "
          f"{np.max(np.abs(mine - iv8)):.1e} V")
else:
    print(f"    brentq との差の最大 {np.max(np.abs(mine - br)):.1e} V")
    print("    （IV-8 の ~/home-lab/partIV/common/nonlinear.py が無いので、IV-8 との照合はとばしました。"
          "IV-8 を済ませたあとで、もう一度実行すると表示されます）")

print("\n4. Rs の効き方（C = 0）")
for u in (0.6, 1.0, 1.5):
    base = cm.newton_static(u, R, d, d)[0]
    row = []
    for Rs in (1.0, 10.0):
        dr = cm.diode(np.log10(2.52e-9), 1.752, Rs, temp_c=25.0)
        row.append((cm.newton_static(u, R, dr, dr)[0] - base) * 1e3)
    i_mA = (u - base) / R * 1e3
    print(f"    u = {u:.1f} V：v = {base:.4f} V、電流 {i_mA:.3f} mA、Rs = 1 Ω で {row[0]:+.3f} mV、10 Ω で {row[1]:+.3f} mV")

print("\n5. 台形則の毎時刻のニュートン法：初期値による反復回数（1 V・312.5 Hz、C = 0.1 µF、刻み 1/640 kHz）")
N, up, f0, C = 512, 4, 312.5, 0.1e-6
vin = np.sin(2 * np.pi * np.arange(N) / N)
u_ = cm._fft_upsample(vin, up)
k = 1.0 / (f0 * N * up) / 2
for name in ("1 つ前の値", "0"):
    v, its, up_ = 0.0, [], u_[-1]
    for per in range(3):
        for m in range(N * up):
            fpv = ((up_ - v) / R - float(cm.pair_iv(v, d, d)[0])) / C
            x = v if name == "1 つ前の値" else 0.0
            for it in range(1, 100):
                i, g = cm.pair_iv(x, d, d)
                gg = x - v - k * (fpv + ((u_[m] - x) / R - float(i)) / C)
                st = float(np.clip(gg / (1 + k * (1 / R + float(g)) / C), -4 * a, 4 * a))
                x -= st
                if abs(st) < 1e-11:
                    break
            its.append(it)
            v, up_ = x, u_[m]
    its = np.array(its[-N * up:])
    print(f"    初期値 {name:6s}：平均 {its.mean():.2f} 回、最大 {its.max()} 回")

In [ ]:
%%writefile home-lab/ch6/synth_clipper.py
"""第6章：実機がなくても解析を試すための合成データ（~/home-lab/ch6/synth_clipper.py）

  clipper.c と同じ条件（PWM 80 kHz・表 256 段・312.5 Hz、ADC0〜2 のラウンドロビン 480 kS/s）で、
  GPIO → R_F1 → C_F1 → R_F2 → C_F2（= vin）→ R → v（C ∥ D1 ∥ D2 → Vb）の回路を、PWM の 1 周期を 48 に分けた
  刻み（0.26 µs）の台形則＋ニュートン法で解き、ADC の時刻で取り出して、量子化と雑音（1.6 mV）を足す。
  真の値（下の TRUE）は解析する側には知らせない。第1章の事後サンプルの代わりも作る。

  uv run python synth_clipper.py --out data_synth [--amps 25,50,…] [--reps 4] [--periods 16] [--seed 6] [--random-phase]
  （--random-phase：実機と同じく、取り込みごとに始まりの位相をばらばらにする。付けなければ毎回 表の先頭から）
  出力：data_synth/cap_<連番>.npz（clipper_capture.py と同じ形）、data_synth/meta.json、
        data_synth/D1_synth_samples.npz・D2_synth_samples.npz（第1章の *_samples.npz と同じ形）
"""
import argparse
import json
import pathlib

import numpy as np

import clipper_model as cm

TOP1, TABLE_N, SUB = 1875, 256, 48          # PWM の段数、表の段数、PWM 1 周期の刻みの数
F_PWM = 150e6 / TOP1                        # 80 kHz
F0 = F_PWM / TABLE_N                        # 312.5 Hz
N_PER, FS = 512, 480e3
TRUE = dict(VOH=3.300, ROUT=40.0, RF1=998.0, CF1=0.104e-6, RF2=1003.0, CF2=0.097e-6, R=2193.0, C=0.0985e-6,
            RB_TOP=10.02e3, RB_BOT=9.97e3, VREF=3.297, NOISE=1.6e-3, T1=22.5, T6=24.0, KAPPA=-2.1e-3,
            D1=(np.log10(2.8e-9), 1.80, 0.7), D2=(np.log10(3.2e-9), 1.83, 0.8))


def table_levels(pm):
    mid = TOP1 // 2
    a = int(round(pm / 1000 * (mid - 1)))
    return mid + np.round(a * np.sin(2 * np.pi * np.arange(TABLE_N) / TABLE_N)).astype(int)


def simulate(pm, voh, p, d1, d2, vb, periods=4):
    """回路の定常の 1 周期を返す（刻みごとの x1・vin・v。電圧は AGND から）"""
    lev = table_levels(pm)
    j = np.arange(SUB)
    frac = np.clip(lev[:, None] / TOP1 * SUB - j[None, :], 0.0, 1.0).ravel()   # 刻みごとの High の割合
    u = voh * frac
    h = 1.0 / (F_PWM * SUB)
    G1, G2, G = 1 / (p["RF1"] + p["ROUT"]), 1 / p["RF2"], 1 / p["R"]
    Cv = np.array([p["CF1"], p["CF2"], p["C"]])

    def f(x, uu):
        i, g = cm.pair_iv(x[2] - vb, d1, d2)
        r = np.array([G1 * (uu - x[0]) - G2 * (x[0] - x[1]),
                      G2 * (x[0] - x[1]) - G * (x[1] - x[2]),
                      G * (x[1] - x[2]) - i]) / Cv
        J = np.array([[-G1 - G2, G2, 0.0], [G2, -G2 - G, G], [0.0, G, -G - g]]) / Cv[:, None]
        return r, J

    x = np.array([vb, vb, vb])
    M = len(u)
    out = np.empty((M, 3))
    up_ = u[-1]
    for per in range(periods):
        start = x.copy()
        for m in range(M):
            fp, _ = f(x, up_)
            y = x.copy()
            for _ in range(30):
                fy, J = f(y, u[m])
                r = y - x - h / 2 * (fp + fy)
                dy = np.linalg.solve(np.eye(3) - h / 2 * J, r)
                y -= dy
                if np.max(np.abs(dy)) < 1e-11:
                    break
            x = y
            out[m] = x
            up_ = u[m]
    return out, float(np.max(np.abs(x - start)))


def fake_posterior(log10_Is, n, Rs, temp_c, rng, size=4000, i_ref=30e-6):
    """第1章の事後サンプルの代わり。I = 30 µA の電圧はよく決まり（sd 0.6 mV）、n は sd 0.03、
    Rs はほとんど決まらない（第1章の結果と同じ性質）。log10 Is と n は強い相関をもつ"""
    a_true = n * cm.vt(temp_c)
    v_ref = a_true * np.log(i_ref / 10 ** log10_Is + 1)
    ns = rng.normal(n + 0.01, 0.03, size)
    vr = rng.normal(v_ref, 0.6e-3, size)
    lis = np.log10(i_ref / (np.exp(vr / (ns * cm.vt(temp_c))) - 1))
    rs = np.exp(rng.normal(np.log(1.0), 0.8, size))
    return dict(log10_Is=lis, n=ns, Rs=rs, temp_C=temp_c)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--out", default="data_synth")
    ap.add_argument("--amps", default="25,50,100,200,300,400,500,600,700,800,900")
    ap.add_argument("--reps", type=int, default=4)
    ap.add_argument("--periods", type=int, default=16)
    ap.add_argument("--seed", type=int, default=6)
    ap.add_argument("--random-phase", action="store_true", help="取り込みの始まりを表のどこにするかを乱数で決める")
    a = ap.parse_args()
    rng = np.random.default_rng(a.seed)
    rng_ph = np.random.default_rng(a.seed + 1)        # 始まりの位相だけの乱数（ほかの乱数の並びを変えない）
    out = pathlib.Path(a.out)
    out.mkdir(parents=True, exist_ok=True)
    p = TRUE
    shift = -p["KAPPA"] * (p["T6"] - p["T1"])
    d1 = cm.diode(*p["D1"], temp_c=p["T1"], shift=shift)
    d2 = cm.diode(*p["D2"], temp_c=p["T1"], shift=shift)
    vb = p["VOH"] * p["RB_BOT"] / (p["RB_TOP"] + p["RB_BOT"])
    for name, dd in (("D1", p["D1"]), ("D2", p["D2"])):
        np.savez(out / f"{name}_synth_samples.npz", **fake_posterior(*dd, p["T1"], rng))
    amps = [int(s) for s in a.amps.split(",")]
    order = [(pm, r) for pm in amps for r in range(a.reps)]
    rng.shuffle(order)
    cache = {}
    m_adc = np.arange(N_PER)                          # 1 周期の中の ADC の番号
    for k, (pm, r) in enumerate(order):
        voh = p["VOH"] * (1 + 2e-4 * rng.standard_normal())  # 取り込みごとの電源のゆらぎ（0.02%、仮定）
        key = (pm, round(voh, 6))
        if key not in cache:
            cache[key] = simulate(pm, voh, p, d1, d2, vb * voh / p["VOH"])
        wave, perr = cache[key]
        M = len(wave)
        s0 = int(rng_ph.integers(M)) if a.random_phase else 0   # 取り込みの始まり（刻みの番号）
        raw = np.empty((a.periods, N_PER, 3))
        for ch in range(3):
            idx = (s0 + (3 * m_adc + ch) * (M // (3 * N_PER))) % M   # ADC の時刻（刻み 8 個ごと）
            x = wave[idx, min(ch, 1) + 1] if ch < 2 else np.full(N_PER, vb * voh / p["VOH"])
            raw[:, :, ch] = x[None, :]
        raw += rng.normal(0, p["NOISE"], raw.shape)
        codes = np.clip(np.round(raw / p["VREF"] * 4096), 0, 4095).astype(np.uint16)
        np.savez(out / f"cap_{k:03d}.npz", raw=codes.reshape(-1), f0=F0, n=N_PER, p=a.periods, fs=FS, amp=pm,
                 rep=r, order=k)
        print(f"cap_{k:03d}  AMP {pm:3d}  rep {r}  始まり {360 * s0 / M:5.1f}°  （周期の差 {perr:.1e} V）")
    meta = dict(source="synth_clipper.py", seed=a.seed, amps=amps, reps=a.reps, vref=p["VREF"], R=p["R"],
                temp_c=p["T6"], random_phase=a.random_phase, note="真の値は synth_clipper.py の TRUE")
    (out / "meta.json").write_text(json.dumps(meta, ensure_ascii=False, indent=1))
    print(f"{len(order)} 個の取り込みを {out}/ に保存しました")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile home-lab/ch6/clipper_analysis.py
"""第6章：クリッパの取り込みを解析し、ニュートン法のモデルと比べる（~/home-lab/ch6/clipper_analysis.py）

  uv run python clipper_analysis.py data/run1 --vref 3.297 --R 2193 --temp 24.0 \
      --post1 ../ch1/data_iv/1N4148_A_samples.npz [--post2 ../ch1/data_iv/1N4148_B_samples.npz] [--plot run1.png]
  （第1章の事後サンプルが無ければ --post1 を付けない。IV-8 の仮定の広い分布を使う）

  1. 取り込みごとに：周期で平均し、ナイキストのビン（PWM のリプルの折り返し）を消し、ADC0・ADC2 の時刻を
     ADC1 にそろえ、vin の基本波が sin の形（位相 0）から始まるように全チャネルを同じ時間だけずらし、Vb から測った vin・v にする。倍音の振幅と THD_F（20 倍音まで）、1 標本の雑音を求める
  2. 出力の振幅が 0.12 V 未満の取り込み（ダイオードがほとんど流れない）で、v/vin = 1/(1 + jωτ) から τ = RC を校正する
  3. それより大きい振幅では、第1章の (log10 Is, n, Rs) の組・τ・R・温度の補正 κ（と --vref-rel を与えれば ADC_VREF の
     目盛り）をばらつかせて（★）を解き、
     THD の予測（雑音の床を 2 乗の和で足したもの）の 95% 区間を、測った THD（取り込みを単位にした 95% 区間）と比べる。
     (誤) n だけを並べ替えて log10 Is との相関を壊した場合の区間も一緒に出す
"""
import argparse
import pathlib

import numpy as np
import pandas as pd
from scipy import stats

import clipper_model as cm

KMAX = 20


def load(folder, vref):
    caps = []
    for f in sorted(pathlib.Path(folder).glob("cap_*.npz")):
        z = np.load(f)
        N, P, fs, f0 = int(z["n"]), int(z["p"]), float(z["fs"]), float(z["f0"])
        x = z["raw"].astype(float).reshape(P, N, 3) * vref / 4096.0
        caps.append(dict(file=f.name, amp=int(z["amp"]), x=x, N=N, P=P, fs=fs, f0=f0))
    if not caps:
        raise SystemExit(f"{folder} に cap_*.npz がありません")
    return caps


def condition(c):
    """周期平均 → ナイキストのビンを消す → 時刻をそろえる → 始まりの位相をそろえる → Vb から測った vin・v"""
    x, N, fs = c["x"], c["N"], c["fs"]
    m = x.mean(axis=0)                                  # (N, 3)：周期で平均（雑音は 1/√P）
    resid = x - m[None]
    c["sigma"] = float(resid[:, :, :2].std(ddof=1) * np.sqrt(c["P"] / (c["P"] - 1)))   # 1 標本の雑音
    X = np.fft.rfft(m, axis=0)
    X[-1] = 0.0                                         # ビン N/2（80 kHz の PWM のリプルが折り返す場所）
    w = 2 * np.pi * np.fft.rfftfreq(N, 1.0 / (c["f0"] * N))
    dt = 1.0 / fs                                       # ラウンドロビン：ADC0 は ADC1 の 1/fs 前、ADC2 は後
    X[:, 0] *= np.exp(1j * w * dt)
    X[:, 2] *= np.exp(-1j * w * dt)
    # 取り込みの始まりは正弦波の表と同期していないので、毎回 位相が違う。vin の基本波の位相 φ を測り、
    # 全チャネルを同じ時間 (φ + π/2)/ω だけずらす（k 倍音は k 倍の位相）。これで vin が sin の形にそろい、平均できる
    phi = np.angle(X[1, 0])
    k = np.arange(X.shape[0])
    X *= np.exp(-1j * k * (phi + np.pi / 2))[:, None]
    c["phase_deg"] = float(np.degrees(phi + np.pi / 2) % 360)
    m = np.fft.irfft(X, n=N, axis=0)
    vb = m[:, 2].mean()
    c["vb"] = float(vb)
    c["vin"] = m[:, 0] - vb
    c["v"] = m[:, 1] - vb
    Ain, Av = cm.harmonics(c["vin"], KMAX), cm.harmonics(c["v"], KMAX)
    c.update(a_in=Ain[1], a_out=Av[1], dc_out=float(c["v"].mean()),
             thd_in=float(cm.thd_f(c["vin"], KMAX)), thd_out=float(cm.thd_f(c["v"], KMAX)),
             h2=Av[2] / Av[1], h3=Av[3] / Av[1])
    # THD の雑音の床：平均した波形の 1 ビンの雑音の振幅 ≈ 2σ/√(N·P)、2〜20 倍音の 19 個ぶん
    c["thd_floor"] = float(np.sqrt(KMAX - 1) * 2 * c["sigma"] / np.sqrt(N * c["P"]) / Av[1])
    return c


def calibrate_tau(caps, lim=0.12):
    """小さい振幅の取り込みで τ = RC を求める。1/H − 1 = jωτ（H = V1/Vin1、ダイオードの電流は無視）"""
    taus, gains = [], []
    for c in caps:
        if c["a_out"] >= lim:
            continue
        X0, X1 = np.fft.rfft(c["vin"])[1], np.fft.rfft(c["v"])[1]
        Hinv = X0 / X1
        taus.append(Hinv.imag / (2 * np.pi * c["f0"]))
        gains.append(Hinv.real - 1.0)
    if len(taus) < 2:
        raise SystemExit(f"出力の振幅が {lim} V 未満の取り込みが 2 個以上要ります（AMP 25〜100 を入れる）")
    t = np.array(taus)
    se = t.std(ddof=1) / np.sqrt(len(t))
    return float(t.mean()), float(se), len(t), float(np.mean(gains))


def draws(a, rng, S):
    if a.post1:
        p1 = cm.load_posterior(a.post1, S, rng)
        p2 = cm.load_posterior(a.post2, S, rng) if a.post2 else p1
        src = "第1章の事後サンプル" + ("（D1・D2 を別々に）" if a.post2 else "（D2 も D1 と同じ値）")
    else:
        p1 = p2 = cm.prior_samples(S, rng)
        src = "IV-8 の仮定の分布（第1章の事後サンプルなし）"
    return p1, p2, src


def make_diodes(p1, p2, kappa, a, perm=None):
    """温度の補正 shift = −κ(T6 − T1) を入れた D1・D2。perm を与えると n だけを並べ替える（誤りの例）"""
    n1 = p1["n"] if perm is None else p1["n"][perm]
    n2 = p2["n"] if perm is None else p2["n"][perm]
    sh1 = -kappa * (a.temp - p1["temp_C"])
    sh2 = -kappa * (a.temp - p2["temp_C"])
    return (cm.diode(p1["log10_Is"], n1, p1["Rs"], temp_c=p1["temp_C"], shift=sh1),
            cm.diode(p2["log10_Is"], n2, p2["Rs"], temp_c=p2["temp_C"], shift=sh2))


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("folder")
    ap.add_argument("--vref", type=float, required=True, help="ADC_VREF の実測値 [V]")
    ap.add_argument("--R", type=float, required=True, help="クリッパの R の実測値 [Ω]")
    ap.add_argument("--r-rel", type=float, default=0.005, help="R の相対誤差（DE-200A の仕様で要確認）")
    ap.add_argument("--temp", type=float, required=True, help="この測定のときの室温 [℃]")
    ap.add_argument("--post1")
    ap.add_argument("--post2")
    ap.add_argument("--kappa", type=float, default=-2.0e-3, help="順電圧の温度係数 [V/℃]（データシートで要確認）")
    ap.add_argument("--kappa-sd", type=float, default=0.3e-3)
    ap.add_argument("--vref-rel", type=float, default=0.0,
                    help="ADC_VREF の目盛りの相対誤差。第1章と同じテスタ・同じレンジで測ったなら 0（打ち消し合う）")
    ap.add_argument("--draws", type=int, default=200)
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--plot")
    a = ap.parse_args()
    rng = np.random.default_rng(a.seed)

    caps = [condition(c) for c in load(a.folder, a.vref)]
    f0, N, P = caps[0]["f0"], caps[0]["N"], caps[0]["P"]
    df = pd.DataFrame([{k: c[k] for k in ("file", "amp", "a_in", "a_out", "thd_in", "thd_out", "thd_floor",
                                          "h2", "h3", "dc_out", "vb", "sigma", "phase_deg")} for c in caps])
    df.to_csv(pathlib.Path(a.folder) / "captures.csv", index=False)
    print(f"取り込み {len(caps)} 個（f0 = {f0:.4f} Hz、1 周期 {N} 点 × {P} 周期）")
    print(f"1 標本の雑音 σ = {df.sigma.median() * 1e3:.2f} mV（中央値）→ ENOB の目安 "
          f"{np.log2(a.vref / (df.sigma.median() * np.sqrt(12))):.1f} ビット、Vb = {df.vb.mean():.4f} V")

    tau, tau_se, n_cal, gain = calibrate_tau(caps)
    tcrit = stats.t.ppf(0.975, n_cal - 1)
    print(f"τ = RC の校正（出力 0.12 V 未満の {n_cal} 個）：τ = {tau * 1e6:.2f} µs "
          f"［{(tau - tcrit * tau_se) * 1e6:.2f}, {(tau + tcrit * tau_se) * 1e6:.2f}］、C = τ/R = {tau / a.R * 1e9:.1f} nF、"
          f"Re(1/H) − 1 = {gain:+.4f}（0 から大きくずれたら ADC0・ADC1 の利得の違いか配線を疑う）")

    S = a.draws
    p1, p2, src = draws(a, rng, S)
    print(f"パラメータの分布：{src}、{S} 組。log10 Is と n の相関 {np.corrcoef(p1['log10_Is'], p1['n'])[0, 1]:+.3f}")
    Rs_ = rng.normal(a.R, a.r_rel * a.R, S)
    tau_s = rng.normal(tau, tau_se, S)
    kap = rng.normal(a.kappa, a.kappa_sd, S)
    d1, d2 = make_diodes(p1, p2, kap, a)
    w1, w2 = make_diodes(p1, p2, kap, a, perm=rng.permutation(S))
    # ADC_VREF の目盛り g：本当の電圧 = 測った電圧 × g。モデルには vin·g を入れ、出力を g で割って測った目盛りに戻す
    g = rng.normal(1.0, a.vref_rel, S) if a.vref_rel > 0 else np.ones(S)

    levels = sorted({c["amp"] for c in caps if c["a_out"] >= 0.12})
    rows, keep = [], {}
    for pm in levels:
        cs = [c for c in caps if c["amp"] == pm]
        vin = np.mean([c["vin"] for c in cs], axis=0)   # 同じ AMP の取り込みの入力の平均（位相をそろえたあと）をモデルに入れる
        vmeas = np.mean([c["v"] for c in cs], axis=0)
        th = np.array([c["thd_out"] for c in cs])
        k = len(th)
        h = stats.t.ppf(0.975, k - 1) * th.std(ddof=1) / np.sqrt(k) if k > 1 else np.nan
        vin_g = vin[None, :] * g[:, None]
        vs, info = cm.simulate_periodic(vin_g, f0, Rs_, tau_s / Rs_, d1, d2)
        vs = vs / g[:, None]
        vw, _ = cm.simulate_periodic(vin_g, f0, Rs_, tau_s / Rs_, w1, w2)
        X = np.fft.rfft(vin)
        X[2:] = 0.0                                     # 入力を「基本波＋直流」だけの正弦波にしたら？
        vp, _ = cm.simulate_periodic(np.fft.irfft(X, n=N)[None, :] * g[:, None], f0, Rs_, tau_s / Rs_, d1, d2)
        tm, tw = cm.thd_f(vs, KMAX), cm.thd_f(vw, KMAX)
        floor = np.mean([c["thd_floor"] for c in cs])
        tn = np.sqrt(tm ** 2 + floor ** 2)               # 測ると雑音の床が足される（2 乗の和で近似）
        lo, hi = np.percentile(tn, [2.5, 97.5])
        wlo, whi = np.percentile(np.sqrt(tw ** 2 + floor ** 2), [2.5, 97.5])
        rms = np.sqrt(np.mean((np.median(vs, axis=0) - vmeas) ** 2))
        rows.append(dict(amp=pm, vin_amp=cm.harmonics(vin)[1], n_cap=k, thd_meas=th.mean(), thd_meas_lo=th.mean() - h,
                         thd_meas_hi=th.mean() + h, thd_model_med=np.median(tm), thd_floor=floor, thd_model_lo=lo, thd_model_hi=hi,
                         thd_wrong_lo=wlo, thd_wrong_hi=whi, thd_pure_sine=np.median(cm.thd_f(vp, KMAX)), thd_in=cm.thd_f(vin, KMAX), rms_mV=rms * 1e3, h2_meas=np.mean([c["h2"] for c in cs]),
                         newton_mean=info["newton_mean"], newton_max=info["newton_max"]))
        keep[pm] = (vin, vmeas, vs, cs)
    res = pd.DataFrame(rows)
    res.to_csv(pathlib.Path(a.folder) / "thd_compare.csv", index=False)
    print("\n AMP  入力[V]  取込  測った THD [%]（95%）     モデル [%]  床 [%]  モデル＋床（95%）     (誤)相関なし [%]  波形の差[mV]  2倍音/基本")
    for r in res.itertuples():
        inside = "○" if (r.thd_model_lo <= r.thd_meas <= r.thd_model_hi) else "×"
        print(f" {r.amp:3d}  {r.vin_amp:6.3f}   {r.n_cap:2d}   {100 * r.thd_meas:7.3f} [{100 * r.thd_meas_lo:7.3f},{100 * r.thd_meas_hi:7.3f}]"
              f"   {100 * r.thd_model_med:7.3f}  {100 * r.thd_floor:5.3f}  [{100 * r.thd_model_lo:7.3f},{100 * r.thd_model_hi:7.3f}] {inside}"
              f"   [{100 * r.thd_wrong_lo:6.3f},{100 * r.thd_wrong_hi:6.3f}]   {r.rms_mV:6.2f}     {r.h2_meas:.4f}")
    print("\n入力のひずみの影響（モデルの中央値）：入力をそのまま入れた THD と、入力を基本波だけの正弦波にした THD")
    for r in res.itertuples():
        print(f"   AMP {r.amp:3d}：入力の THD {100 * r.thd_in:6.3f} %、出力 {100 * r.thd_model_med:7.3f} % → 正弦波なら {100 * r.thd_pure_sine:7.3f} %")
    print(f"\nニュートン法の反復：平均 {res.newton_mean.mean():.2f} 回、最大 {res.newton_max.max()} 回（1 刻みあたり）")
    print("○＝測った THD の平均が「モデル＋床」の 95% 区間に入る。(誤) は n を並べ替えて log10 Is との相関を壊したもの（床込み）")

    if a.plot:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        matplotlib.rcParams["font.family"] = ["Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic", "sans-serif"]   # ノートブック版：Colab でも止まらないように候補を足す
        matplotlib.rcParams["axes.unicode_minus"] = False
        fig, ax = plt.subplots(2, 2, figsize=(12, 8.5))
        t_ms = np.arange(N) / (f0 * N) * 1e3
        big = levels[-1]
        vin, vmeas, vs, cs = keep[big]
        ax[0, 0].plot(t_ms, vin, color="tab:gray", label="vin（測定）")
        ax[0, 0].plot(t_ms, vmeas, color="tab:blue", label="v（測定）")
        ax[0, 0].fill_between(t_ms, *np.percentile(vs, [2.5, 97.5], axis=0), color="tab:red", alpha=0.35,
                              label="v（モデル 95%）")
        ax[0, 0].set(xlabel="時刻 [ms]（vin の基本波の位相 0 から）", ylabel="Vb からの電圧 [V]", title=f"波形（AMP {big}）")
        ax[0, 0].legend(fontsize=8)
        for pm in levels[::max(1, len(levels) // 4)]:
            vin, vmeas, vs, _ = keep[pm]
            ax[0, 1].plot(vin, vmeas, ".", ms=2, label=f"AMP {pm}")
        ax[0, 1].set(xlabel="vin [V]", ylabel="v [V]", title="入出力の関係（輪の太さ＝C の記憶）")
        ax[0, 1].legend(fontsize=8)
        x = res.vin_amp
        ax[1, 0].errorbar(x, 100 * res.thd_meas, yerr=100 * (res.thd_meas_hi - res.thd_meas), fmt="o", color="k",
                          label="測定（取り込みを単位に 95%）")
        ax[1, 0].fill_between(x, 100 * res.thd_model_lo, 100 * res.thd_model_hi, color="tab:red", alpha=0.35,
                              label="モデル＋雑音の床（95%）")
        ax[1, 0].plot(x, 100 * res.thd_wrong_lo, "--", color="tab:purple", lw=1, label="(誤) 相関を壊した 95%")
        ax[1, 0].plot(x, 100 * res.thd_wrong_hi, "--", color="tab:purple", lw=1)
        fl = df.groupby("amp").thd_floor.mean()
        ax[1, 0].plot(df.groupby("amp").a_in.mean(), 100 * fl, ":", color="tab:gray", label="雑音の床")
        ax[1, 0].set(xscale="log", yscale="log", xlabel="入力の基本波の振幅 [V]", ylabel="THD_F [%]",
                     title="THD と入力の大きさ")
        ax[1, 0].legend(fontsize=8)
        vin, vmeas, vs, _ = keep[big]
        k = np.arange(1, KMAX + 1)
        Am = cm.harmonics(vmeas, KMAX)[1:]
        Ap = np.median(cm.harmonics(vs, KMAX)[:, 1:], axis=0)
        ax[1, 1].bar(k - 0.2, 20 * np.log10(Am / Am[0]), width=0.4, label="測定")
        ax[1, 1].bar(k + 0.2, 20 * np.log10(Ap / Ap[0]), width=0.4, label="モデル（中央値）")
        ax[1, 1].axhline(20 * np.log10(fl[big] / np.sqrt(KMAX - 1)), color="tab:gray", ls=":", label="1 ビンの雑音の目安")
        ax[1, 1].set(xlabel="倍音の番号 k", ylabel="基本波に対する大きさ [dB]", title=f"倍音（AMP {big}）", ylim=(-110, 5))
        ax[1, 1].legend(fontsize=8)
        for x_ in ax.ravel():
            x_.grid(True, alpha=0.3, which="both")
        fig.tight_layout()
        fig.savefig(a.plot, dpi=130)
        print(f"図を保存しました：{a.plot}")


if __name__ == "__main__":
    main()

In [ ]:
import os, re, shlex, subprocess, sys
import numpy as np
import pandas as pd
from IPython.display import Image, display

CH6 = os.path.abspath("home-lab/ch6")   # ~/home-lab/ch6 の代わり
sys.path.insert(0, CH6)


def run(args):
    """ターミナルの「uv run python ○○.py …」（~/home-lab/ch6 で）の代わり。表示を出し、あとで確かめられるように文字列で返す"""
    p = subprocess.run([sys.executable] + shlex.split(args), capture_output=True, text=True, cwd=CH6)
    print(p.stdout, end="")
    err = "\n".join(l for l in p.stderr.splitlines() if "findfont" not in l)   # 候補のうち見つからないフォントの警告は出さない（日本語のフォントがあるかは、0-1 のセルの表示で分かる）
    if err.strip():
        print(err, file=sys.stderr)
    assert p.returncode == 0, "★実行に失敗：" + args
    return p.stdout


def show(*paths, width=900):
    """保存された図を表示する（home-lab/ch6 から見た場所）"""
    for f in paths:
        print(f)
        display(Image(filename=os.path.join(CH6, f), width=width))


def ok(cond, msg):
    assert cond, "★確かめ失敗：" + msg
    print("OK", msg)


def near(got, want, d):
    """got が want と ±d 以内か（d は表示の最後の桁の 1 つ分。numpy・scipy の版で最後の桁が変わる分の幅）。リストなら要素ごと（d もリストにできる）"""
    if isinstance(want, (list, tuple)):
        got = list(got)
        ds = d if isinstance(d, (list, tuple)) else [d] * len(want)
        return len(got) == len(want) and all(near(g, w, e) for g, w, e in zip(got, want, ds))
    return abs(float(got) - want) <= d * (1 + 1e-6)


def num(pattern, text, i=1):
    """表示（text）から、pattern の (…) の数を取り出す"""
    m = re.search(pattern, text)
    assert m, "表示に見つからない：" + pattern
    return float(m.group(i).replace("−", "-"))


SYN = "data_synth --R 2193 --temp 24.0"                   # run_ch6.sh synth と同じ R・室温
POSTS = "--post1 data_synth/D1_synth_samples.npz --post2 data_synth/D2_synth_samples.npz"


def analyze(args):
    """clipper_analysis.py を実行し、表示と、表（data_synth/thd_compare.csv）を返す。
    inside は表示の「○」と同じ（測った THD の平均が「モデル＋床」の 95% 区間に入る）。幅はポイント（%）で"""
    out = run("clipper_analysis.py " + args)
    t = pd.read_csv(os.path.join(CH6, "data_synth/thd_compare.csv")).set_index("amp")
    t["inside"] = (t.thd_model_lo <= t.thd_meas) & (t.thd_meas <= t.thd_model_hi)
    t["width"] = 100 * (t.thd_model_hi - t.thd_model_lo)
    t["width_wrong"] = 100 * (t.thd_wrong_hi - t.thd_wrong_lo)
    marks = re.findall(r"^ +(\d+) .*\] ([○×]) ", out, re.M)
    assert [int(a) for a, _ in marks] == list(t.index) and [m == "○" for _, m in marks] == list(t.inside), "表示と CSV が合わない"
    return out, t


print("準備ができました：", CH6)

## 1. ニュートン法の収束を確かめる（6-3-4、手順 A の 1）

`run_ch6.sh synth` の1行目です（数秒）。$C = 0$ の式 $f(v) = v - u + R\,i_D(v) = 0$ を、初期値と方法（plain＝そのまま、limit＝1 回に動く量を $\pm 4nV_T$ までに制限、line＝減衰つきニュートン法）を変えて解きます。表の「−1」は失敗（値があふれた・200 回で止まらない）です。初期値 40 V では $\exp(40/0.045)$ が倍精度の最大を超えるので、どの方法でも失敗するのが正しい動きです。

In [ ]:
out_nd = run("newton_demo.py")

In [ ]:
# 確かめ（手順 A の 1 と 6-3-4）
rows = {(float(u), float(v0)): (int(a), int(b), int(c)) for u, v0, a, b, c in
        re.findall(r"^ +(-?[\d.]+) +(-?[\d.]+) +(-?\d+) +(-?\d+) +(-?\d+) +(?:[\d.]+|nan)$", out_nd, re.M)}
ok(len(rows) == 12, "1. の表が 12 行")
ok(rows[(3.0, 0.0)] == (60, 10, 7), "u = 3.0・v0 = 0.0 の行が plain 60・limit 10・line 7 回")
ok(rows[(1.0, 0.0)] == (17, 9, 6), "u = 1.0・v0 = 0.0 は plain 17 回・line 6 回（確認問題 3 の答え）")
ok(all(rows[(u, 40.0)] == (-1, -1, -1) for u in (0.3, 1.0, 3.0)), "初期値 40 V はどの方法でも失敗（−1）")
ok(rows[(0.3, 2.0)] == (40, 40, 40) and rows[(1.0, 2.0)] == (39, 39, 39) and rows[(3.0, 2.0)] == (37, 37, 37)
   and rows[(3.0, 3.0)] == (59, 59, 59), "指数の領域の奥から出ると、3 つの方法の回数は同じ（v0 = 2 V で 37〜40 回、u = 3・v0 = 3 で 59 回）")
ok(abs(num(r"K = f''/\(2f'\) = ([\d.]+)", out_nd) - 10.17) < 0.005, "K = f''/(2f') = 10.17 /V")
ratio = [float(x) for x in re.findall(r"e_\(k\+1\)/e_k² = +([\d.]+)", out_nd)]
ok(len(ratio) >= 4 and all(10.0 <= r <= 11.0 for r in ratio[1:5]), f"2 次収束の比 e_(k+1)/e_k² が約 10（10.1〜10.9）：{ratio[1:5]}")
d_br = num(r"brentq との差の最大 ([\d.e+-]+) V", out_nd)
d_iv8 = num(r"clipper_static_newton との差の最大 ([\d.e+-]+) V", out_nd)
ok(d_br <= 1e-12 and d_iv8 <= 1e-12, f"照合の差が 1e-12 V 以下（brentq {d_br:.1e} V、IV-8 {d_iv8:.1e} V。教科書は 3.3e-16・1.1e-16）")
ok(near(num(r"u = 1.0 V：v = ([\d.]+) V", out_nd), 0.5125, 0.0001), "u = 1 V の解 v = 0.5125 V")
m5 = re.findall(r"初期値 (.+?)\s*：平均 ([\d.]+) 回、最大 (\d+) 回", out_nd)
ok([(float(a), int(b)) for _, a, b in m5] == [(2.98, 3), (5.20, 6)],
   "台形則の毎時刻の初期値：1 つ前の値なら平均 2.98 回・最大 3 回、0 なら平均 5.2 回・最大 6 回")

## 2. 6-3-1 の数値例と、IV-8 の台形則との照合（6-3-1、6-3-4 の (5)）

`clipper_model.py` を import して、本文の数値例を計算します（IV-8 の見本の値 $I_s = 2.52$ nA、$n = 1.752$、25 ℃、$R_s = 0$、$R = 2.2$ kΩ）。$C = 0$ の頭打ちの電圧と、$C = 0.1\,\mu$F で 312.5 Hz の正弦波を入れたときの THD_F です（図6-4）。最後に、312.5 Hz・1 V の正弦波の定常解を、IV-8 の `nl.clipper_trapezoid`（同じ刻み 1/640 kHz、同じ台形則）と比べます。このセルは本文のコードではなく、本文の値を確かめるために書き足したものです（数秒）。

In [ ]:
import clipper_model as cm

d = cm.diode(np.log10(2.52e-9), 1.752, 0.0, temp_c=25.0)
vs_static = [cm.newton_static(u, 2.2e3, d, d)[0] for u in (0.5, 1, 2, 3)]
print("C = 0：vin = 0.5, 1, 2, 3 V で v =", ", ".join(f"{v:.4f}" for v in vs_static), "V")
N = 512
thd = {}
for A in (0.1, 0.2, 0.3, 0.5, 1.0, 1.5):
    v, info = cm.simulate_periodic(A * np.sin(2 * np.pi * np.arange(N) / N), 312.5, 2.2e3, 0.1e-6, d, d)
    thd[A] = 100 * float(cm.thd_f(v))
    print(f"C = 0.1 µF、振幅 {A:.1f} V：THD_F = {thd[A]:.4f} %（{info['periods']} 周期で定常、ニュートン法 平均 {info['newton_mean']:.2f} 回）")

v1, info1 = cm.simulate_periodic(np.sin(2 * np.pi * np.arange(N) / N), 312.5, 2.2e3, 0.1e-6, d, d)
sys.path.insert(0, os.path.abspath("home-lab/partIV"))
from common import nonlinear as nl   # IV-8 の共通ライブラリ（照合の相手。第6章のファイルは import しない）
p = nl.clipper_params(R=2.2e3, C=0.1e-6, Is=2.52e-9, n=1.752, VT=float(cm.vt(25.0)))
M = N * 4
vt_, _ = nl.clipper_trapezoid(np.sin(2 * np.pi * np.arange(M * info1["periods"]) / M), 312.5 * M, p)
diff_tr = np.max(np.abs(vt_[-M:][::4] - v1))
print(f"IV-8 の clipper_trapezoid（同じ刻み、同じ周期の数）との差の最大 {diff_tr:.1e} V")

In [ ]:
# 確かめ（6-3-1 の数値例と 6-3-4 の (5)）
ok(near(vs_static, [0.4270, 0.5125, 0.5612, 0.5845], 0.0001), "C = 0 の v = 0.4270・0.5125・0.5612・0.5845 V（頭打ち）")
ok(near([thd[A] for A in thd], [0.0034, 0.027, 0.17, 2.9, 14.8, 20.6], [0.0001, 0.001, 0.01, 0.1, 0.1, 0.1]),
   "THD_F は振幅 0.1・0.2・0.3・0.5・1.0・1.5 V で 0.0034・0.027・0.17・2.9・14.8・20.6 %")
ok(diff_tr < 1e-12, f"IV-8 の台形則との差 {diff_tr:.1e} V（同じ式なので丸めの誤差の程度。教科書は 1.1e-16 V）")

## 3. 合成データを作る（Pico B の取り込みの代わり。手順 A の 1）

`run_ch6.sh synth` の2行目です。`synth_clipper.py` は、`clipper.c` と同じ条件（PWM 80 kHz・表 256 段・312.5 Hz、ADC0〜2 のラウンドロビン 480 kS/s）で、GPIO → RC 2 段 → クリッパの回路を細かい刻みの台形則＋ニュートン法で解き、ADC の量子化と雑音（1.6 mV）を足して、11 通りの振幅（`AMP` 25〜900）× 4 回 = 44 個の取り込みを `data_synth/` に作ります。第1章の事後サンプルの代わり（`D1_synth_samples.npz`・`D2_synth_samples.npz`）も作ります。真の値（`TRUE`）は、解析する側には知らせません。

**このセルがいちばん重く、執筆時の Mac で約2分かかります**（Colab ではその数倍）。教科書の 6-7 の値と比べるため、設定は小さくしていません（取り込みの数を減らすと、乱数の順が変わって 6-7 の値と比べられなくなります）。

In [ ]:
out_syn = run("synth_clipper.py --out data_synth")   # 教科書と同じ（--amps 25〜900 の 11 通り × --reps 4、--seed 6）

In [ ]:
# 確かめ
caps = sorted(f for f in os.listdir(os.path.join(CH6, "data_synth")) if f.startswith("cap_"))
ok(len(caps) == 44 and caps[0] == "cap_000.npz" and caps[-1] == "cap_043.npz", "cap_000.npz〜cap_043.npz の 44 個ができた")
ok(all(os.path.exists(os.path.join(CH6, "data_synth", f)) for f in ("meta.json", "D1_synth_samples.npz", "D2_synth_samples.npz")),
   "meta.json と、事後サンプルの代わり（D1・D2）ができた")
perr = [float(x) for x in re.findall(r"周期の差 ([\d.e+-]+) V", out_syn)]
ok(len(perr) == 44 and max(perr) < 1e-6, f"回路の計算は定常になっている（周期の始めと終わりの差の最大 {max(perr):.1e} V）")
z = np.load(os.path.join(CH6, "data_synth/D1_synth_samples.npz"))
r = np.corrcoef(z["log10_Is"], z["n"])[0, 1]
ok(r > 0.99, f"合成の事後サンプルでも log10 Is と n は強く相関している（{r:+.3f}。第1章と同じ性質）")

## 4. 合成データの解析（手順 A の 1・E の 10〜11、6-7）

`run_ch6.sh synth` の3行目です（執筆時の Mac で約25秒）。`clipper_analysis.py` は、取り込みごとに周期で平均してナイキストのビン（PWM のリプルの折り返し）を消し、ラウンドロビンの時刻と、取り込みの始まりの位相（`clipper.c` は表の先頭を待たずに取り込みを始めるので、毎回違う。教科書 6-3-3）をそろえてから、

1. 1 標本の雑音 σ と ENOB の目安、Vb を出し、
2. 出力の振幅が 0.12 V 未満の取り込み（`AMP` 25〜100）で $\tau = RC$ を校正し、
3. それより大きい振幅（`AMP` 200〜900）で、第1章の $(\log_{10}I_s, n, R_s)$ の組・τ・R・温度の補正 κ をばらつかせて（★）を解き、THD の予測（雑音の床を2乗の和で足したもの）の 95% 区間を、測った THD と比べます。(誤) の列は、$n$ だけを並べ替えて $\log_{10}I_s$ との相関を壊したときの区間です。

ここでは `--post1`・`--post2` に、3 節で作った合成の事後サンプル（第1章の `1N4148_A_…_samples.npz`・`1N4148_B_…_samples.npz` の代わり）を渡します。

In [ ]:
out_main, t_main = analyze(f"{SYN} --vref 3.297 {POSTS} --plot data_synth/synth.png")
show("data_synth/synth.png")

In [ ]:
# 確かめ（手順 A の 1・E の 10〜11 と 6-7）
ok(abs(num(r"雑音 σ = ([\d.]+) mV", out_main) - 1.62) < 0.005 and near(num(r"ENOB の目安 ([\d.]+) ビット", out_main), 9.2, 0.1),
   "1 標本の雑音 σ = 1.62 mV（入れた 1.6 mV）、ENOB の目安 9.2 ビット")
ok(abs(num(r"Vb = ([\d.]+) V", out_main) - 1.6458) < 0.00005, "Vb = 1.6458 V（PWM の中心 1.6491 V より 3 mV 低い）")
tau = num(r"τ = ([\d.]+) µs", out_main)
lo, hi = [float(x) for x in re.search(r"µs ［([\d.]+), ([\d.]+)］", out_main).groups()]
ok(num(r"出力 0.12 V 未満の (\d+) 個", out_main) == 12 and abs(tau - 215.93) < 0.02 and abs(lo - 215.64) < 0.02 and abs(hi - 216.21) < 0.02,
   f"τ の校正（AMP 25〜100 の 12 個）：{tau} µs ［{lo}, {hi}］（教科書 215.93 ［215.64, 216.21］）。真の値 216.0 µs が区間に入る")
ok(lo <= 216.0 <= hi and near(num(r"C = τ/R = ([\d.]+) nF", out_main), 98.5, 0.1), "真の τ = 216.0 µs が区間に入り、C = 98.5 nF")
ok(abs(num(r"Re\(1/H\) − 1 = ([-+\d.]+)", out_main)) <= 0.001, "Re(1/H) − 1 = −0.0002（±0.01 以内）")
ok(abs(num(r"相関 ([-+\d.]+)", out_main) - 0.997) < 0.002, "log10 Is と n の相関 +0.997")
ok(list(t_main.index) == [200, 300, 400, 500, 600, 700, 800, 900] and t_main.inside.all(),
   "比べる対象の AMP 200〜900（出力 0.12 V 以上）の 8 行がすべて「○」")
ok(near(100 * t_main.thd_meas, [0.107, 0.400, 1.952, 4.985, 7.978, 10.466, 12.546, 14.253], 0.001),
   "測った THD は 0.107・0.400・1.952・4.985・7.978・10.466・12.546・14.253 %（6-7）")
ok(near(t_main.vin_amp, [0.230, 0.346, 0.458, 0.565, 0.666, 0.761, 0.854, 0.945], 0.001),
   "入力の基本波 0.230〜0.945 V")
ok(near((100 * t_main.thd_model_lo)[[200, 500, 900]], [0.106, 4.914, 14.114], 0.001)
   and near((100 * t_main.thd_model_hi)[[200, 500, 900]], [0.121, 5.035, 14.334], 0.001),
   "モデル＋床の 95% 区間：AMP 200 ［0.106, 0.121］、500 ［4.914, 5.035］、900 ［14.114, 14.334］（6-7 の表）")
ok(near(t_main.rms_mV.min(), 0.21, 0.01) and near(t_main.rms_mV.max(), 0.49, 0.01), "波形の差（モデルの中央値と測定の RMS）は 0.21〜0.49 mV")
ok(near(num(r"反復：平均 ([\d.]+) 回", out_main), 2.93, 0.01) and num(r"最大 (\d+) 回（1 刻み", out_main) == 3, "ニュートン法は 1 刻みあたり平均 2.93 回・最大 3 回")
rw = t_main.width_wrong / t_main.width
ok(3.5 < rw[900] < 4.5 and 10 < rw[400] < 14 and 1.5 < rw[200] < 2.5 and all(rw[[300, 400, 500, 600, 700, 800, 900]] > 3.5),
   f"(誤) 相関を壊すと区間の幅が AMP 300〜900 で約 4〜12 倍（900 で {rw[900]:.1f} 倍、400 で {rw[400]:.1f} 倍、200 は {rw[200]:.1f} 倍）")
ok(near(100 * t_main.h2_meas.min(), 0.04, 0.01) and near(100 * t_main.h2_meas.max(), 0.18, 0.01), "2 倍音は基本波の 0.04〜0.18 %")

## 4-2. 取り込みの始まりの位相がばらばらなとき（6-3-3、手順 A の 1）

`run_ch6.sh synth` の4〜5行目です。3〜4 節の合成データは、毎回 正弦波の表の先頭から取り込んだことになっているので、`clipper_analysis.py` の「始まりの位相をそろえる」処理は、実質なにもしていません。ところが実機の `clipper.c` は表の先頭を待たずに取り込みを始めるので、**位相は取り込みごとに必ず違います**（6-3-3）。そこで `--random-phase` を付けて、始まりを取り込みごとの乱数にした合成データを、別のフォルダ `data_synth_rp/` に作り（3〜4 節の `data_synth/` とは混ぜない）、同じ解析を通します。

**合成データを作るのに、執筆時の Mac で約2分、解析に約25秒かかります。** 最後に、そろえずに平均したら入力がどう見えるかを、`AMP` 900 の 4 回で計算して比べます（このセルは本文のコードではなく、6-3-3 の値を確かめるために書き足したものです）。

In [ ]:
out_syn_rp = run("synth_clipper.py --out data_synth_rp --random-phase")   # 種は同じ（--seed 6）。始まりの位相だけを乱数にする
phases = [float(x) for x in re.findall(r"始まり +([\d.]+)°", out_syn_rp)]
print("始まりの位相（44 回）：", ", ".join(f"{p:.0f}" for p in phases[:8]), "…")

In [ ]:
out_rp = run("clipper_analysis.py data_synth_rp --vref 3.297 --R 2193 --temp 24.0 "
             "--post1 data_synth_rp/D1_synth_samples.npz --post2 data_synth_rp/D2_synth_samples.npz --plot data_synth_rp/synth_rp.png")
t_rp = pd.read_csv(os.path.join(CH6, "data_synth_rp/thd_compare.csv")).set_index("amp")
t_rp["inside"] = (t_rp.thd_model_lo <= t_rp.thd_meas) & (t_rp.thd_meas <= t_rp.thd_model_hi)

# そろえずに平均したら：AMP 900 の 4 回の vin（ADC0）を、周期で平均してから、そのまま 4 回で平均する
import clipper_analysis as ca
c900 = [c for c in ca.load(os.path.join(CH6, "data_synth_rp"), 3.297) if c["amp"] == 900]
m_raw = np.mean([c["x"].mean(axis=0)[:, 0] for c in c900], axis=0)
a_raw = 2 * np.abs(np.fft.rfft(m_raw - m_raw.mean())[1]) / len(m_raw)
a_al = np.mean([ca.condition(c)["a_in"] for c in c900])
print(f"AMP 900 の入力の基本波：そろえずに平均 {a_raw:.3f} V、そろえてから平均（解析と同じ） {a_al:.3f} V")
print("4 回の始まりの位相：", [round(c["phase_deg"]) for c in c900], "度")

In [ ]:
# 確かめ（6-3-3）
ok(len(phases) == 44 and np.ptp(phases) > 180, f"始まりの位相が取り込みごとにばらばら（{min(phases):.0f}〜{max(phases):.0f}°）")
ok(near(a_al, 0.94, 0.01) and 0.40 < a_raw < 0.55,
   f"そろえずに平均すると、AMP 900 の入力が 0.94 V のはずが約半分（{a_raw:.2f} V）に見える。そろえると {a_al:.3f} V（教科書は 0.47 V と 0.94 V）")
ok(list(t_rp.index) == [200, 300, 400, 500, 600, 700, 800, 900] and t_rp.inside[[300, 400, 500, 600, 700, 800, 900]].all(),
   "位相をそろえた後は AMP 300〜900 が「○」")
print("AMP 200：", "○" if t_rp.inside[200] else "×", f"（測った {100 * t_rp.thd_meas[200]:.3f} %、区間［{100 * t_rp.thd_model_lo[200]:.3f}, {100 * t_rp.thd_model_hi[200]:.3f}］）"
      "。教科書では × で、雑音の床に近いところの多重比較の × の 1 つ（6-7）。乱数の版によっては ○ になることもある")
ok(near(t_rp.vin_amp.min(), 0.23, 0.01) and near(t_rp.vin_amp.max(), 0.945, 0.01), "入力の振幅は 0.23〜0.94 V（3〜4 節と同じ）")
ok(near(t_rp.rms_mV.min(), 0.21, 0.02) and near(t_rp.rms_mV.max(), 0.48, 0.02), "波形の差は 0.21〜0.48 mV（6-3-3）")
ok(near(100 * t_rp.thd_meas, list(100 * t_main.thd_meas), [0.005, 0.01, 0.01, 0.01, 0.02, 0.02, 0.03, 0.03]),
   "測った THD は、位相を乱数にしても 4 節とほぼ同じ（差は 0.03 ポイント以内）")

## 5. 第1章の事後サンプルが無いとき（6-7 の (c)、`prior_samples`）

第1章で 1N4148 を測っていないときは、`run_ch6.sh analyze` は `--post1`・`--post2` を付けずに `clipper_analysis.py` を実行します（「事後サンプル：（なし。IV-8 の仮定の分布を使う）」と表示される）。そのとき `clipper_model.py` の `prior_samples` が、IV-8 の発展課題と同じ仮定の広い分布（$\log_{10}I_s \sim \mathcal N(\log_{10}2.52\times10^{-9}, 0.3^2)$、$n \sim \mathcal N(1.75, 0.05^2)$、$R_s$ は中央 0.5 Ω の対数正規。独立）を使います。D1・D2 は同じ値になります。同じ合成データで試します（約25秒）。

In [ ]:
out_prior, t_prior = analyze(f"{SYN} --vref 3.297")   # run_ch6.sh analyze で *_samples.npz が無いときと同じ（--post1・--post2 なし）

In [ ]:
# 確かめ（6-7 の (c)）
ok("IV-8 の仮定の分布（第1章の事後サンプルなし）" in out_prior and abs(num(r"相関 ([-+\d.]+)", out_prior)) < 0.2,
   "prior_samples を使った（log10 Is と n はほぼ無相関）")
ok(t_prior.inside.all(), "AMP 200〜900 のすべてで「○」")
ok(near(100 * t_prior.thd_model_lo[900], 12.5, 0.1) and near(100 * t_prior.thd_model_hi[900], 16.3, 0.1),
   f"ただし AMP 900 の区間は {100 * t_prior.thd_model_lo[900]:.1f}〜{100 * t_prior.thd_model_hi[900]:.1f} %（事後サンプルのときは 14.1〜14.3 %）")
k = t_prior.width[900] / t_main.width[900]
ok(16 < k < 19, f"区間の幅が事後サンプルのときの約 17 倍（{k:.1f} 倍）。「区間に入った」は区間が広ければいつでも言えるので、幅も一緒に報告する")

## 6. ADC_VREF の目盛りの不確かさを伝播させる（6-3-6、`--vref-rel`）

ADC_VREF を第1章と同じテスタ・同じレンジで測れなかったときは、`--vref-rel 0.005` を付けて、目盛り $g \sim \mathcal N(1, 0.005^2)$ をばらつかせます（モデルの入力を $v_{\mathrm{in}} g$ にし、出力を $g$ で割って戻す。測った THD は目盛りによらない）。区間がどれだけ広がるかを、4 節（`--vref-rel` なし）と比べます（約25秒）。

In [ ]:
out_rel, t_rel = analyze(f"{SYN} --vref 3.297 {POSTS} --vref-rel 0.005")

In [ ]:
# 確かめ（6-3-6）
ok(t_rel.inside.all(), "--vref-rel 0.005 でも AMP 200〜900 のすべてで「○」")
k = t_rel.width / t_main.width
ok(near(t_main.width[900], 0.22, 0.01) and near(t_rel.width[900], 0.36, 0.01) and 1.5 < k[900] < 1.8,
   f"AMP 900 で区間の幅が約 1.6 倍（0.22 → 0.36 ポイント、{k[900]:.2f} 倍）")
ok(2.4 < k[400] < 3.4 and 2.4 < k[500] < 3.4 and k.max() < 3.4, f"AMP 400〜500 では約 3 倍まで（{k[400]:.1f}・{k[500]:.1f} 倍）")

## 7. （任意）条件を変えたとき（6-7 の (a)(b)、6-3-6 の ADC_VREF のずれ）

6 回の解析で、執筆時の Mac で約2分半かかるので、**既定ではとばします**（`EXTRA = False`）。試すときは、下のセルの `EXTRA = False` を `EXTRA = True` に書きかえて実行してください。

- (a) D2 を測らずに D1 の値で代用（`--post2` なし）
- (b) 温度の補正をしない（`--temp` を第1章の測定と同じ 22.5 ℃ にする。合成データでは第1章より 1.5 ℃ 高い室温で取り込んだことになっている）
- ADC_VREF を +0.5 %・−0.5 % ずらして測ったとき（`--vref 3.313485`・`3.280515`）と、そこに `--vref-rel 0.005` を足したとき

In [ ]:
EXTRA = False   # True にすると 6 回の解析をする（執筆時の Mac で約2分半）
if EXTRA:
    out_a, t_a = analyze(f"{SYN} --vref 3.297 --post1 data_synth/D1_synth_samples.npz")   # (a) D2 も D1 と同じ値
    out_b, t_b = analyze(f"data_synth --R 2193 --temp 22.5 --vref 3.297 {POSTS}")         # (b) TEMP を第1章と同じに
    out_p, t_p = analyze(f"{SYN} --vref 3.313485 {POSTS}")                              # ADC_VREF を +0.5 %
    out_m, t_m = analyze(f"{SYN} --vref 3.280515 {POSTS}")                              # ADC_VREF を −0.5 %
    out_pr, t_pr = analyze(f"{SYN} --vref 3.313485 {POSTS} --vref-rel 0.005")
    out_mr, t_mr = analyze(f"{SYN} --vref 3.280515 {POSTS} --vref-rel 0.005")
else:
    print("とばしました（EXTRA = False）")

In [ ]:
# 確かめ（6-7 の (a)(b) と 6-3-6）
if EXTRA:
    out_ = lambda t: [a for a in t.index if not t.inside[a]]
    ok("D2 も D1 と同じ値" in out_a and out_(t_a) == [200] and near(t_a.rms_mV.max(), 0.75, 0.01),
       "(a) D2 を D1 で代用：波形の差が 0.75 mV に増え、AMP 200 で区間が狭すぎて外れた")
    dthd = 100 * (t_main.thd_model_med - t_b.thd_model_med)
    ok(out_(t_b) == [300, 400, 500, 600, 700, 800, 900] and near(t_b.rms_mV.max(), 2.46, 0.01) and all(0.07 < dthd[[500, 700, 900]]) and all(dthd[[500, 700, 900]] < 0.15),
       f"(b) 温度の補正なし：AMP 300 以上のすべてで外れ（200 は入る）、モデルの THD が約 0.1 ポイント低く（{dthd[900]:.2f}）、波形の差 2.46 mV")
    ok(out_(t_p) == [400, 500] and out_(t_m) == [400, 500, 600, 700, 800]
       and near(t_p.rms_mV.max(), 1.07, 0.01) and near(t_m.rms_mV.max(), 1.91, 0.01),
       "--vref +0.5 % で AMP 400・500 が、−0.5 % で AMP 400〜800 が区間の外。波形の差は最大 1.07・1.91 mV")
    ok(all(t_pr.inside[[300, 400, 500, 600, 700, 800, 900]]) and t_mr.inside.all(),
       "--vref-rel 0.005 を足すと、+0.5 % なら AMP 300〜900、−0.5 % なら 200〜900 がすべて「○」に戻る")
else:
    print("とばしました（EXTRA = False）")

## 8. 読むだけ：Pico B のプログラム（clipper.c）

教科書の 6-6-1 の `clipper.c` と `CMakeLists.txt` の変更点です。**ここでは実行もビルドもしません**（Colab には Pico をつなげません）。ビルドと書き込みは、Mac の VS Code の Raspberry Pi Pico 拡張で、教科書の手順 B の 4 のとおりに行ってください（執筆時に pico-sdk 2.3.1、Board は pico2_w で、`-Wall -Wextra` を付けても警告なしでビルドできることを確かめてあります）。取り込みの `clipper_capture.py`（`run_ch6.sh check`・`run`）も、Pico B とシリアルで話すので入れていません。3 節の `synth_clipper.py` が、この `clipper.c` と同じ形の `cap_*.npz` を作る代わりです。

```c
// 第6章：ダイオードクリッパの入出力を記録する（Pico B / C SDK）
//
//   GP15（ピン20）：PWM 80kHz。DMA が 1 周期ごとに正弦波の表から次のデューティを書き込む（DDS）。
//                  RC 2 段で平滑すると 312.5Hz の正弦波（中心 約1.65V）になる。
//   GP14（ピン19）：バイアス用の分圧（10kΩ＋10kΩ＋470µF）の電源。ON で High、OFF で Low。
//   ADC0（GP26）＝ 入力 vin、ADC1（GP27）＝ 出力 v、ADC2（GP28）＝ バイアス Vb を、
//   ラウンドロビン＋DMA で 480kS/s（各チャネル 160kS/s ＝ 1 周期にちょうど 512 点）で記録し、
//   12 ビットの値を USB シリアルに 16 進数 3 桁で送る。解析は Mac の clipper_analysis.py で行う。
//
// シリアルコマンド（1 行ずつ。応答の先頭の文字：I＝情報、E＝誤り）
//   PING → PONG
//   INFO → I,info,clk_sys[Hz],clk_adc[Hz],f0[Hz],fs[Hz]
//   ON   → バイアスを入れ、PWM を 50%（振幅 0）にする → I,on
//   AMP a（0〜950、最大振幅の千分率）→ I,amp,a,振幅のカウント
//   RUN p（記録する周期の数 1〜16）→ F,f0,N,P,fs,a の行、16 進数のデータ行、最後に END
//   STAT → I,stat,ADC0,ADC1,ADC2（各 256 回の平均。12 ビットの値）
//   OFF  → 振幅 0・PWM 0%・バイアス Low（USB を抜く前に必ず送り、15 秒待つ）→ I,off
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <math.h>
#include "pico/stdlib.h"
#include "hardware/adc.h"
#include "hardware/clocks.h"
#include "hardware/dma.h"
#include "hardware/gpio.h"
#include "hardware/pwm.h"

#define SINE_PIN 15                  // PWM の出力（ピン20）。スライス 7 のチャネル B
#define BIAS_PIN 14                  // バイアスの分圧の電源（ピン19）。普通の GPIO として使う
#define PWM_TOP 1874u                // カウンタは 0〜1874 の 1875 段：150MHz / 1875 = 80kHz
#define TABLE_N 256u                 // 正弦波 1 周期の段数：80kHz / 256 = 312.5Hz
#define N_PER 512                    // 1 周期・1 チャネルあたりの標本数
#define N_CH 3                       // ADC0・ADC1・ADC2
#define MAX_PER 16                   // 1 回に記録できる周期の数
#define BUF_N (N_CH * N_PER * MAX_PER)
#define ADC_DIV 99.0f                // 48MHz / (1 + 99) = 480kS/s（3 チャネル合計）
#define AMP_MAX_PM 950               // 振幅の上限（千分率）。0V・3.3V に張り付かない余裕

// DMA のリング（アドレスの下位ビットだけが回る）を使うため、表は「大きさの倍数」の番地に置く
static uint32_t table[TABLE_N] __attribute__((aligned(TABLE_N * sizeof(uint32_t))));
static uint16_t buf[BUF_N];          // 記録用（48KB）。大きいので関数の中ではなく、ここに置く
static int dds_chan = -1;            // 正弦波の表 → PWM の DMA チャネル
static int adc_chan = -1;            // ADC → buf の DMA チャネル
static uint slice;                   // GP15 の PWM スライス（7）
static bool on = false;
static int amp_pm = 0;

static const uint32_t MID = (PWM_TOP + 1u) / 2u;   // 937（デューティ 約50% ＝ 約1.65V）

// 表を作り直す。CC レジスタの上位 16 ビットがチャネル B（GP15）、下位 16 ビットがチャネル A（GP14。
// GP14 は PWM ではなく普通の GPIO にしているので、下位の値は出力に関係しない）
static int fill_table(int pm) {
    int a = (int)lroundf((float)pm / 1000.0f * (float)(MID - 1u));
    for (uint32_t k = 0; k < TABLE_N; k++) {
        float s = sinf(2.0f * (float)M_PI * (float)k / (float)TABLE_N);
        int level = (int)MID + (int)lroundf((float)a * s);
        table[k] = (uint32_t)level << 16;
    }
    return a;
}

static void dds_start(void) {
    dma_channel_config c = dma_channel_get_default_config(dds_chan);
    channel_config_set_transfer_data_size(&c, DMA_SIZE_32);
    channel_config_set_read_increment(&c, true);
    channel_config_set_write_increment(&c, false);
    channel_config_set_ring(&c, false, 10);           // 読む番地の下位 10 ビット（1024 バイト＝256 語）で回る
    channel_config_set_dreq(&c, pwm_get_dreq(slice)); // PWM のカウンタが一巡するたびに 1 語
    dma_channel_configure(dds_chan, &c, &pwm_hw->slice[slice].cc, table,
                          dma_encode_endless_transfer_count(), true);   // 終わりなく続ける（RP2350）
}

static void all_off(void) {
    // RP2350 のエラッタ E5：abort の前にチャネルの EN ビットを消しておかないと、また動き出すことがある
    // （pico-sdk の dma.h の dma_channel_abort() の注記）
    hw_clear_bits(&dma_hw->ch[dds_chan].al1_ctrl, DMA_CH0_CTRL_TRIG_EN_BITS);
    dma_channel_abort(dds_chan);
    pwm_set_chan_level(slice, PWM_CHAN_B, 0);         // 入力側を 0V へ
    gpio_put(BIAS_PIN, 0);                            // バイアス側も 0V へ（470µF は分圧の抵抗で放電）
    on = false;
    amp_pm = 0;
}

static void capture(int periods) {
    int count = N_CH * N_PER * periods;
    adc_run(false);
    adc_fifo_drain();
    adc_set_clkdiv(ADC_DIV);
    adc_select_input(0);                              // ADC0 から始めて 0, 1, 2, 0, 1, 2, …
    adc_set_round_robin(0x7u);                        // ビット 0・1・2 ＝ ADC0〜2
    adc_fifo_setup(true, true, 1, false, false);      // FIFO を使い、1 個たまるたびに DMA に知らせる

    dma_channel_config c = dma_channel_get_default_config(adc_chan);
    channel_config_set_transfer_data_size(&c, DMA_SIZE_16);
    channel_config_set_read_increment(&c, false);
    channel_config_set_write_increment(&c, true);
    channel_config_set_dreq(&c, DREQ_ADC);
    dma_channel_configure(adc_chan, &c, buf, &adc_hw->fifo, (uint)count, true);
    adc_run(true);
    dma_channel_wait_for_finish_blocking(adc_chan);
    adc_run(false);
    adc_fifo_drain();
    adc_set_round_robin(0);

    float f0 = (float)clock_get_hz(clk_sys) / (float)(PWM_TOP + 1u) / (float)TABLE_N;
    float fs = (float)clock_get_hz(clk_adc) / (1.0f + ADC_DIV);
    printf("F,%.4f,%d,%d,%.2f,%d\n", f0, N_PER, periods, fs, amp_pm);
    for (int i = 0; i < count; i++) {
        printf("%03x", buf[i] & 0xfffu);
        if (i % 48 == 47 || i == count - 1) printf("\n");
    }
    printf("END\n");
}

static void stat(void) {
    float m[N_CH];
    for (int ch = 0; ch < N_CH; ch++) {
        adc_select_input((uint)ch);
        uint32_t s = 0;
        for (int i = 0; i < 256; i++) s += adc_read();
        m[ch] = (float)s / 256.0f;
    }
    printf("I,stat,%.1f,%.1f,%.1f\n", m[0], m[1], m[2]);
}

int main(void) {
    stdio_init_all();
    gpio_init(BIAS_PIN);
    gpio_set_dir(BIAS_PIN, GPIO_OUT);
    gpio_put(BIAS_PIN, 0);

    gpio_set_function(SINE_PIN, GPIO_FUNC_PWM);
    slice = pwm_gpio_to_slice_num(SINE_PIN);
    pwm_config pc = pwm_get_default_config();
    pwm_config_set_clkdiv_int(&pc, 1);                // 分周しない（150MHz で数える）
    pwm_config_set_wrap(&pc, PWM_TOP);
    pwm_init(slice, &pc, true);
    pwm_set_chan_level(slice, PWM_CHAN_B, 0);

    adc_init();
    adc_gpio_init(26);                                // デジタル入力を切って ADC 用にする（E9 対策にもなる）
    adc_gpio_init(27);
    adc_gpio_init(28);
    dds_chan = dma_claim_unused_channel(true);
    adc_chan = dma_claim_unused_channel(true);

    char line[32];
    int len = 0;
    while (true) {
        int ch = getchar_timeout_us(100000);
        if (ch == PICO_ERROR_TIMEOUT) continue;
        if (ch != '\r' && ch != '\n') {
            if (len < 31) line[len++] = (char)ch;
            continue;
        }
        line[len] = 0;
        len = 0;
        if (strcmp(line, "PING") == 0) {
            printf("PONG\n");
        } else if (strcmp(line, "INFO") == 0) {
            uint32_t fsys = clock_get_hz(clk_sys), fadc = clock_get_hz(clk_adc);
            printf("I,info,%lu,%lu,%.4f,%.2f\n", (unsigned long)fsys, (unsigned long)fadc,
                   (float)fsys / (float)(PWM_TOP + 1u) / (float)TABLE_N, (float)fadc / (1.0f + ADC_DIV));
        } else if (strcmp(line, "ON") == 0) {
            if (!on) {
                fill_table(0);
                dds_start();
                gpio_put(BIAS_PIN, 1);
                on = true;
            }
            printf("I,on\n");
        } else if (strcmp(line, "OFF") == 0) {
            all_off();
            printf("I,off\n");
        } else if (strncmp(line, "AMP ", 4) == 0) {
            int pm = atoi(line + 4);
            if (!on) {
                printf("E,send ON first\n");
            } else if (pm < 0 || pm > AMP_MAX_PM) {
                printf("E,AMP must be 0..%d\n", AMP_MAX_PM);
            } else {
                int a = fill_table(pm);
                amp_pm = pm;
                printf("I,amp,%d,%d\n", pm, a);
            }
        } else if (strncmp(line, "RUN ", 4) == 0) {
            int p = atoi(line + 4);
            if (!on) {
                printf("E,send ON first\n");
            } else if (p < 1 || p > MAX_PER) {
                printf("E,RUN must be 1..%d\n", MAX_PER);
            } else {
                capture(p);
            }
        } else if (strcmp(line, "STAT") == 0) {
            stat();
        } else if (line[0] != 0) {
            printf("E,unknown: %s\n", line);
        }
    }
}
```

`CMakeLists.txt` の変更点：

```cmake
# VS Code の Raspberry Pi Pico 拡張で「New C/C++ Project」（Board: Pico 2 W、名前: clipper、場所: ~/home-lab/ch6）を作り、
# clipper.c を上のコードに置き換えてから、CMakeLists.txt の該当部分を次のようにする
target_link_libraries(clipper
        pico_stdlib
        hardware_adc
        hardware_dma
        hardware_pwm)

# USBシリアルを有効にする（なければ足す）
pico_enable_stdio_usb(clipper 1)
pico_enable_stdio_uart(clipper 0)
```

## おわりに

- ニュートン法では、初期値が悪いと指数の領域から 1 回に約 $nV_T$ ずつしか戻れないこと、減衰つきニュートン法と電圧の制限が「行きすぎ」を防ぐこと、解の近くでは 2 次収束すること、台形則では 1 つ前の値を初期値にすれば 3 回ほどで収束することを確かめました。自分で書いたモデルは、brentq と IV-8 の `nonlinear.py` で照合しました。
- 合成データの解析では、小さい振幅で τ を校正し、大きい振幅で、事後サンプルを組のまま回路モデルに通した THD の区間が、測った THD を含むこと（6-7 の表と同じ値）を見ました。相関を壊すと区間が広がりすぎること、第1章の事後サンプルが無いと（`prior_samples`）区間が約 17 倍広くなること、ADC_VREF の目盛りの不確かさ（`--vref-rel`）で区間がどれだけ広がるかも確かめました。実機と同じく取り込みの始まりの位相がばらばらでも（`--random-phase`）、位相をそろえてから平均すれば同じ結果に戻ること（そろえないと入力が約半分に見えること）も確かめました。
- このノートブックに入れていないもの：Pico B のプログラムのビルドと書き込み、配線、取り込み（手順 B〜D）、第1章の 2 本の測定（手順 A の 2）、自分の取り込みの解析（手順 E の `bash run_ch6.sh analyze run1`）。これらは Mac で教科書の手順のとおりに行います。実機では合成データより合わないことのほうが多く、合わないこと自体を原因の候補とともに記録するのが結果です（6-7）。